# Boop — KataGo-Style AlphaZero Training

Trains a **CNN ResNet policy+value network** with MCTS self-play,
incorporating key improvements from KataGo:

| Improvement | Vanilla AlphaZero | This notebook |
|---|---|---|
| Network | MLP (3×256) | **6-block ResNet + SE attention**, CNN on 6×6 board |
| Input | Flat 184-float obs | **9-channel (5 board + 4 hand)** 6×6 spatial tensor |
| Training data | 1× per move | **8× via all 8 board symmetries** |
| MCTS sims | Fixed | **Playout cap randomization** (15 fast / 100 full) |
| Leaf eval | 1 leaf / forward pass | **Batched-leaf MCTS** (16 leaves / forward pass via virtual loss) |
| Self-play | One game at a time | **N parallel games** — leaf waves of all games share one forward pass |
| Temperature | Constant | **Annealing** — 1.0 first 30 moves, 0.05 after |
| LR | Cosine from ep 1 | **Linear warmup** (50 eps) then cosine decay |
| Optimizer | Adam | **AdamW** (true weight decay) |
| Entropy | None | **Entropy bonus** (prevents early mode collapse) |

### Network architecture
```
Input (9, 6, 6)  →  Stem (Conv 3×3, GroupNorm, ReLU)
                 →  6 × ResBlock (Conv-GN-ReLU-Conv-GN + SE) + skip
                 →  Policy head: 1×1 conv (2 ch) → flatten → Linear(108)
                 →  Value head:  1×1 conv (1 ch) → flatten → Linear(128) → ReLU → Linear(1) → Tanh
```

### KataGo improvements in detail
- **SE attention**: per-channel gating learns which feature maps matter for each position
- **Symmetry augmentation**: 4 rotations × 2 reflections = 8× free training data
- **Playout cap randomization**: 75% fast (15 sim) games for diversity + 25% full (100 sim) games for quality
- **Temperature annealing per game**: exploratory early, near-greedy for endgame decisions
- **LR warmup**: avoids destabilising BatchNorm layers with large early gradients
- **Entropy bonus**: small regulariser that keeps the policy spread out early in training
- **Forced playouts + policy-target pruning (KataGo)**: noise-seeded root moves are guaranteed sqrt(k·P·N) visits during self-play, and visits they did not earn are removed from the policy target — the policy stops learning to imitate Dirichlet noise (a direct attack on the KL floor).
- **Multiprocess self-play + GPU inference server** (this branch): CPU worker processes do all game/tree work; one server batches every worker's NN requests into large forward passes on the GPU (DirectML only wins with big batches). Small-batch eval runs on CPU replicas.
- **Parallel self-play + vectorized engine**: N games run concurrently and their leaf waves are evaluated in one joint forward pass; win/promotion checks, legal moves and the observer are vectorized over a precomputed line table, and `clone()` is a fast field copy (~4x higher self-play throughput end-to-end).
- **Batched-leaf MCTS**: collects a wave of leaves with *virtual loss*, then evaluates them all in one NN forward pass instead of one at a time. This is KataGo's core speedup; the gain scales with hardware parallelism (modest on CPU, large on GPU).

In [ ]:
%pip install open_spiel -q
# AMD/Intel GPU acceleration via DirectML (Windows / WSL2). Needs Python
# 3.11 or 3.12 (torch-directml has no 3.13 wheel) and pins torch==2.3.1.
# Harmless if you're on CUDA/CPU — the device code below falls back.
%pip install torch-directml -q

In [ ]:
# Boop board game -- inlined so Colab works without the custom package.
# Faithful rules: graduation conserves pieces (kittens BECOME cats), and a
# player whose pool is empty must graduate a kitten of their choice rather than
# being stuck (which previously caused spurious draws).
# Perf: win checks, promotion, legal moves and the observer are vectorized over
# a precomputed 3-in-a-row line table, and clone() is a hand-rolled field copy
# instead of deepcopy — MCTS clones a state once per simulation, so these are
# the hottest paths in the whole pipeline.

import numpy as np
from open_spiel.python.observation import IIGObserverForPublicInfoGame
import pyspiel

_NUM_PLAYERS = 2
_ROWS = 6
_COLS = 6
_NUM_CELLS = _ROWS * _COLS
_NUM_PIECE_TYPES = 2
_NUM_PIECES = 8                              # each player has exactly 8 pieces
_GRADUATE_OFFSET = _NUM_PIECE_TYPES * _NUM_CELLS  # 72: graduation actions start here
_NUM_ACTIONS = _GRADUATE_OFFSET + _NUM_CELLS      # 108 = 72 placement + 36 graduate
_MAX_KITTENS = 8
_MAX_CATS = 8                               # all 8 pieces can become cats
_MAX_GAME_LENGTH = 500

_EMPTY = 0
_P0_KITTEN = 1
_P0_CAT = 2
_P1_KITTEN = 3
_P1_CAT = 4

_KITTEN_VAL = [_P0_KITTEN, _P1_KITTEN]
_CAT_VAL = [_P0_CAT, _P1_CAT]
_PIECE_VALS = [[_P0_KITTEN, _P0_CAT], [_P1_KITTEN, _P1_CAT]]

# Every 3-in-a-row line on the board (horizontal, vertical, both diagonals) as
# indices into the flattened 36-cell board: shape (80, 3). Win detection and
# promotion become single vectorized gathers instead of triple Python loops.
_LINE_IDX = np.array(
    [[(r + k * dr) * _COLS + (c + k * dc) for k in range(3)]
     for r in range(_ROWS) for c in range(_COLS)
     for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1))
     if 0 <= r + 2 * dr < _ROWS and 0 <= c + 2 * dc < _COLS],
    dtype=np.int64)

_GAME_TYPE = pyspiel.GameType(
    short_name='python_boop',
    long_name='Python Boop',
    dynamics=pyspiel.GameType.Dynamics.SEQUENTIAL,
    chance_mode=pyspiel.GameType.ChanceMode.DETERMINISTIC,
    information=pyspiel.GameType.Information.PERFECT_INFORMATION,
    utility=pyspiel.GameType.Utility.ZERO_SUM,
    reward_model=pyspiel.GameType.RewardModel.TERMINAL,
    max_num_players=_NUM_PLAYERS,
    min_num_players=_NUM_PLAYERS,
    provides_information_state_string=True,
    provides_information_state_tensor=False,
    provides_observation_string=True,
    provides_observation_tensor=True,
    parameter_specification={})

_GAME_INFO = pyspiel.GameInfo(
    num_distinct_actions=_NUM_ACTIONS,
    max_chance_outcomes=0,
    num_players=_NUM_PLAYERS,
    min_utility=-1.0,
    max_utility=1.0,
    utility_sum=0.0,
    max_game_length=_MAX_GAME_LENGTH)


class BoopGame(pyspiel.Game):
  def __init__(self, params=None):
    super().__init__(_GAME_TYPE, _GAME_INFO, params or dict())

  def new_initial_state(self):
    return BoopState(self)

  def make_py_observer(self, iig_obs_type=None, params=None):
    if ((iig_obs_type is None) or
        (iig_obs_type.public_info and not iig_obs_type.perfect_recall)):
      return BoopObserver(params)
    return IIGObserverForPublicInfoGame(iig_obs_type, params)


class BoopState(pyspiel.State):
  def __init__(self, game):
    super().__init__(game)
    self._game_ref = game
    self._cur_player = 0
    self._is_terminal = False
    self._winner = None
    self._move_count = 0
    self.board = np.zeros((_ROWS, _COLS), dtype=np.int8)
    self._hand = [[_MAX_KITTENS, 0], [_MAX_KITTENS, 0]]

  def clone(self):
    # Fast clone: MCTS calls this once per simulation. Copying the handful of
    # fields directly is ~10x cheaper than the default deepcopy-based clone.
    cp = BoopState(self._game_ref)
    cp._cur_player  = self._cur_player
    cp._is_terminal = self._is_terminal
    cp._winner      = self._winner
    cp._move_count  = self._move_count
    cp.board        = self.board.copy()
    cp._hand        = [self._hand[0][:], self._hand[1][:]]
    return cp

  def current_player(self):
    return pyspiel.PlayerId.TERMINAL if self._is_terminal else self._cur_player

  def _legal_actions(self, player):
    # Forced-graduation rule: if the pool is empty, the player must graduate one
    # of their kittens on the board (returning it to the pool as a cat) instead
    # of placing. Graduation actions are _GRADUATE_OFFSET + cell.
    flat = self.board.reshape(-1)
    hk, hc = self._hand[player]
    if hk == 0 and hc == 0:
      kittens = np.flatnonzero(flat == _KITTEN_VAL[player])
      return (_GRADUATE_OFFSET + kittens).tolist()
    empty = np.flatnonzero(flat == _EMPTY)
    if hk > 0 and hc > 0:
      return np.concatenate([empty, _NUM_CELLS + empty]).tolist()
    if hk > 0:
      return empty.tolist()
    return (_NUM_CELLS + empty).tolist()

  def _apply_action(self, action):
    p = self._cur_player
    if action >= _GRADUATE_OFFSET:
      # Forced graduation: a kitten on the board becomes a cat in the pool.
      cell = action - _GRADUATE_OFFSET
      r, c = cell // _COLS, cell % _COLS
      self.board[r, c] = _EMPTY
      self._hand[p][1] += 1
      self._move_count += 1
      self._post_move(p)        # no piece placed → no boop
      return
    piece_type = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    self._hand[p][piece_type] -= 1
    self.board[r, c] = _PIECE_VALS[p][piece_type]
    self._boop(r, c, is_cat=(piece_type == 1))
    self._move_count += 1
    self._post_move(p)

  def _post_move(self, p):
    """Shared post-move resolution: win checks, promotion, turn handoff."""
    if self._move_count >= _MAX_GAME_LENGTH:
      self._is_terminal = True
      return
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._promote_kittens(p)
    self._promote_kittens(1 - p)
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._cur_player = 1 - p
    # With forced graduation a player is never permanently stuck: an empty pool
    # forces a graduation, and a board of eight cats is already a win. The guard
    # below is defensive only and should not trigger in normal play.
    if not self._legal_actions(self._cur_player):
      self._is_terminal = True

  def _action_to_string(self, player, action):
    if action >= _GRADUATE_OFFSET:
      cell = action - _GRADUATE_OFFSET
      r, c = cell // _COLS, cell % _COLS
      return f'p{player}:graduate@({r},{c})'
    pt = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    piece = 'cat' if pt else 'kitten'
    return f'p{player}:{piece}@({r},{c})'

  def is_terminal(self):
    return self._is_terminal

  def returns(self):
    if self._winner == 0:
      return [1.0, -1.0]
    if self._winner == 1:
      return [-1.0, 1.0]
    return [0.0, 0.0]

  def __str__(self):
    syms = {
        _EMPTY: '.', _P0_KITTEN: 'k', _P0_CAT: 'K',
        _P1_KITTEN: 'o', _P1_CAT: 'O',
    }
    rows = [
        ''.join(syms[self.board[r, c]] for c in range(_COLS))
        for r in range(_ROWS)
    ]
    rows.append(
        f'P0: {self._hand[0][0]}k {self._hand[0][1]}K  '
        f'P1: {self._hand[1][0]}k {self._hand[1][1]}K  '
        f'move={self._move_count}')
    return '\n'.join(rows)

  def _boop(self, r, c, is_cat):
    board = self.board
    for dr in (-1, 0, 1):
      for dc in (-1, 0, 1):
        if dr == 0 and dc == 0:
          continue
        nr, nc = r + dr, c + dc
        if not (0 <= nr < _ROWS and 0 <= nc < _COLS):
          continue
        neighbor = board[nr, nc]
        if neighbor == _EMPTY:
          continue
        neighbor_is_cat = neighbor == _P0_CAT or neighbor == _P1_CAT
        if not is_cat and neighbor_is_cat:
          continue
        dest_r, dest_c = nr + dr, nc + dc
        owner = 0 if (neighbor == _P0_KITTEN or neighbor == _P0_CAT) else 1
        n_type = 1 if neighbor_is_cat else 0
        if not (0 <= dest_r < _ROWS and 0 <= dest_c < _COLS):
          board[nr, nc] = _EMPTY
          self._hand[owner][n_type] += 1
        elif board[dest_r, dest_c] == _EMPTY:
          board[dest_r, dest_c] = neighbor
          board[nr, nc] = _EMPTY

  def _promote_kittens(self, player):
    # Faithful rule (rulebook p.4): a line of 3 of the player's own pieces —
    # kittens AND/OR cats mixed — graduates. Every kitten in the line becomes
    # a cat; every cat in the line simply returns to the pool; either way all
    # three board cells clear and the pool gains 3 cats (pieces conserved).
    # A pure 3-cats-in-a-row is a WIN, checked before this runs, so it never
    # reaches here as a live case.
    kv, cv = _KITTEN_VAL[player], _CAT_VAL[player]
    flat = self.board.reshape(-1)
    while True:
      mine = (flat == kv) | (flat == cv)
      if int(mine.sum()) < 3:
        return
      full = mine[_LINE_IDX].all(axis=1)
      if not full.any():
        return
      # Resolve ONE qualifying line per pass, chosen UNIFORMLY AT RANDOM
      # among all lines that currently qualify (not the player's choice —
      # that would need a new action type, which isn't worth the added
      # action-space complexity). Clearing the chosen line's cells
      # invalidates any OVERLAPPING line, so it is not picked again this
      # call — matching "choose one, leave the rest" for a connected run of
      # 4+ (fig.4). An independent (non-overlapping) line elsewhere still
      # qualifies and is resolved on the loop's next pass.
      candidates = _LINE_IDX[full]
      line = candidates[np.random.randint(len(candidates))]
      flat[line] = _EMPTY
      self._hand[player][1] += 3

  def _check_win(self, player):
    flat = self.board.reshape(-1)
    cats = flat == _CAT_VAL[player]
    n = int(cats.sum())
    if n >= _NUM_PIECES:        # win condition 1: all eight pieces are cats
      return True
    if n < 3:
      return False
    # Win condition 2: three cats in a row (orthogonal or diagonal).
    return bool(cats[_LINE_IDX].all(axis=1).any())


class BoopObserver:
  def __init__(self, params):
    if params:
      raise ValueError(f'Observation parameters not supported; passed {params}')
    board_size = 5 * _ROWS * _COLS
    self.tensor = np.zeros(board_size + 4, np.float32)
    self.dict = {
        'observation': np.reshape(self.tensor[:board_size], (5, _ROWS, _COLS)),
        'hand': self.tensor[board_size:],
    }

  def set_from(self, state, player):
    self.tensor.fill(0)
    obs = self.dict['observation']
    hand = self.dict['hand']
    b = state.board
    opp = 1 - player
    obs[0][b == _EMPTY] = 1.0
    obs[1][b == _KITTEN_VAL[player]] = 1.0
    obs[2][b == _CAT_VAL[player]] = 1.0
    obs[3][b == _KITTEN_VAL[opp]] = 1.0
    obs[4][b == _CAT_VAL[opp]] = 1.0
    hand[0] = state._hand[player][0] / _MAX_KITTENS
    hand[1] = state._hand[player][1] / _MAX_CATS
    hand[2] = state._hand[opp][0] / _MAX_KITTENS
    hand[3] = state._hand[opp][1] / _MAX_CATS

  def string_from(self, state, player):
    del player
    return str(state)


pyspiel.register_game(_GAME_TYPE, BoopGame)


In [ ]:
game = pyspiel.load_game('python_boop')
state = game.new_initial_state()
print('Game:', game.get_type().long_name)
print('Actions:', game.num_distinct_actions())
print('Obs size:', game.observation_tensor_size())
print()
print(state)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import random

# ── Device selection ──────────────────────────────────────────────────────────
# This branch uses the GPU the only way DirectML wins: BIG batches. The
# multiprocessing self-play pool funnels every worker's NN requests into one
# server that evaluates them in large joint batches on the GPU, and training
# runs batch-512 steps there too. Small-batch paths (eval games, tournaments)
# run on CPU replicas instead, so DirectML per-op latency stays off the hot path.
DEVICE_PREFERENCE = 'directml' # 'cpu' | 'directml' | 'cuda' | 'auto'

def _pick_device(pref):
    if pref in ('directml', 'auto'):
        try:
            import torch_directml
            try:    _name = torch_directml.device_name(0)
            except Exception: _name = 'DirectML GPU'
            print(f'Using DirectML: {_name}')
            return torch_directml.device(), 'directml'
        except Exception:
            if pref == 'directml':
                print('DirectML requested but unavailable — falling back.')
    if pref in ('cuda', 'auto') and torch.cuda.is_available():
        return torch.device('cuda'), 'cuda'
    return torch.device('cpu'), 'cpu'

DEVICE, _BACKEND = _pick_device(DEVICE_PREFERENCE)


# ── Input helpers ──────────────────────────────────────────────────────────────────────────────

def _obs_to_9ch(obs_np):
    """Flat 184-float obs → (9, 6, 6): 5 board planes + 4 hand scalars broadcast."""
    board = obs_np[..., :180].reshape(*obs_np.shape[:-1], 5, 6, 6)
    hand  = obs_np[..., 180:]
    # broadcast hand scalars spatially so the CNN sees them at every cell
    hand_planes = np.broadcast_to(
        hand[..., None, None], hand.shape + (6, 6)).copy()
    return np.concatenate([board, hand_planes], axis=-3)   # (..., 9, 6, 6)


def state_to_tensor(state, device):
    """Single game state → (1, 9, 6, 6) float tensor."""
    obs = np.array(state.observation_tensor(state.current_player()), dtype=np.float32)
    x   = _obs_to_9ch(obs)[None]        # (1, 9, 6, 6)
    return torch.from_numpy(x).to(device)


def batch_to_tensor(obs_list, device):
    """List of flat 184-float observations → (B, 9, 6, 6) float tensor."""
    obs = np.stack(obs_list).astype(np.float32)   # (B, 184)
    x   = _obs_to_9ch(obs)                         # (B, 9, 6, 6)
    return torch.from_numpy(x).to(device)


# ── Network modules ────────────────────────────────────────────────────────────────────────────

class _GroupNorm(nn.Module):
    """GroupNorm built from elementwise ops (reshape/mean/var/affine) rather than
    torch's fused native_group_norm. Mathematically identical to nn.GroupNorm and
    keeps NO running stats (train == eval), but avoids the fused kernel whose
    backward is broken on DirectML (it raised "NativeBatchNormBackward0 returned
    an invalid gradient" for the 1-channel value-head norm)."""
    def __init__(self, num_groups, num_channels, eps=1e-5):
        super().__init__()
        self.num_groups = num_groups
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))

    def forward(self, x):
        n, c = x.shape[0], x.shape[1]
        xg = x.reshape(n, self.num_groups, -1)          # group channels+spatial
        mean = xg.mean(dim=2, keepdim=True)
        var = (xg - mean).pow(2).mean(dim=2, keepdim=True)
        xg = (xg - mean) / torch.sqrt(var + self.eps)
        x = xg.reshape(x.shape)
        return x * self.weight.view(1, c, 1, 1) + self.bias.view(1, c, 1, 1)


def _norm(channels):
    """Normalizer with no running stats (train == eval) so the value head can't be
    miscalibrated between self-play (eval) and training the way BatchNorm was.
    Uses the hand-rolled GroupNorm above to stay DirectML-safe. `groups` divides
    `channels`."""
    groups = min(8, channels)
    while channels % groups != 0:
        groups -= 1
    return _GroupNorm(groups, channels)


class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention (KataGo-style)."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels * 2),
        )

    def forward(self, x):
        s = x.mean(dim=(2, 3))             # global avg pool → (B, C)
        scale, bias = self.fc(s).chunk(2, dim=1)
        scale = torch.sigmoid(scale)
        return (x * scale[:, :, None, None]
                  + bias[:, :, None, None])


class ResBlock(nn.Module):
    """Residual block: Conv-GN-ReLU-Conv-GN + SE attention + skip."""
    def __init__(self, channels, use_se=True):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            _norm(channels),
        )
        self.se  = SEBlock(channels) if use_se else nn.Identity()
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.se(self.net(x)) + x)


class BoopNet(nn.Module):
    """
    KataGo-style network for Boop.

    Input  : (B, 9, 6, 6) — 5 board planes + 4 hand scalars broadcast
    Body   : Conv stem → N × ResBlock(channels, SE)
    Policy : 1×1 conv (2 ch) → flatten → Linear(_NUM_ACTIONS=108)
    Value  : 1×1 conv (1 ch) → flatten → Linear(128) → ReLU → Linear(1) → Tanh
    """
    def __init__(self, channels=128, num_blocks=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(9, channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
        )
        self.body = nn.Sequential(*[ResBlock(channels) for _ in range(num_blocks)])

        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 2, 1, bias=False),
            _norm(2),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(2 * 6 * 6, _NUM_ACTIONS),
        )
        self.value_head = nn.Sequential(
            nn.Conv2d(channels, 1, 1, bias=False),
            _norm(1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(36, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.body(self.stem(x))
        return self.policy_head(x), self.value_head(x)


print(f'Device: {DEVICE}')
_demo = BoopNet()
print(f'BoopNet params: {sum(p.numel() for p in _demo.parameters()):,}')
del _demo

In [ ]:
from open_spiel.python.algorithms import mcts as mcts_lib


# ── 8-fold board symmetry augmentation ─────────────────────────────────────────
# The 6×6 Boop board has 8 dihedral symmetries (4 rotations × 2 reflections).
# Every self-play sample is augmented with all 8 copies for free training data.

def _compute_action_perm(sym_idx):
    """
    Precompute the 108-element action permutation for board symmetry sym_idx.
    Covers both placement actions (2 piece types x 36 cells) and the 36
    graduation actions, which transform spatially the same way as a cell.
    Symmetries 0-3: 0°/90°/180°/270° CCW rotation.
    Symmetries 4-7: horizontal flip followed by the same rotations.
    """
    k    = sym_idx % 4
    flip = sym_idx >= 4
    perm = np.empty(_NUM_ACTIONS, dtype=np.int32)
    for pt in range(2):
        for r in range(6):
            for c in range(6):
                nr, nc = r, c
                if flip:
                    nc = 5 - nc
                for _ in range(k):       # k × 90° CCW: (r,c) → (5-c, r)
                    nr, nc = 5 - nc, nr
                perm[pt * 36 + r * 6 + c] = pt * 36 + nr * 6 + nc
    for r in range(6):                   # graduation actions (cell-indexed)
        for c in range(6):
            nr, nc = r, c
            if flip:
                nc = 5 - nc
            for _ in range(k):
                nr, nc = 5 - nc, nr
            perm[_GRADUATE_OFFSET + r * 6 + c] = _GRADUATE_OFFSET + nr * 6 + nc
    return perm

_ACTION_PERMS = [_compute_action_perm(i) for i in range(8)]


def augment_sample(obs_flat, policy, value):
    """Return all 8 symmetry copies of a (obs, policy, value) training sample."""
    obs   = np.asarray(obs_flat, dtype=np.float32)
    board = obs[:180].reshape(5, 6, 6)
    hand  = obs[180:]                       # hand counts: spatially invariant
    pol   = np.asarray(policy, dtype=np.float32)

    out = []
    for sym_idx in range(8):
        k    = sym_idx % 4
        flip = sym_idx >= 4
        b    = board.copy()
        if flip:
            b = b[:, :, ::-1]             # horizontal flip (cols)
        b = np.rot90(b, k=k, axes=(1, 2)) # k × 90° CCW
        aug_obs = np.concatenate([b.reshape(-1).copy(), hand])
        # SCATTER the policy (aug[perm[a]] = pol[a]) so it rotates the SAME way
        # as the board. np.rot90 sends (r,c)->(5-c,r), the exact map perm builds,
        # so GATHERING (pol[perm]) would rotate the policy the wrong way for the
        # two non-involutive syms (rot90/rot270), mispairing board and policy on
        # 2 of every 8 augmented samples.
        aug_pol = np.zeros_like(pol)
        aug_pol[_ACTION_PERMS[sym_idx]] = pol
        out.append((aug_obs, aug_pol, value))
    return out


# ── MCTS evaluator ─────────────────────────────────────────────────────────────

class NNEvaluator(mcts_lib.Evaluator):
    """
    Wraps BoopNet as an OpenSpiel MCTS evaluator.
    Per-search cache: evaluate() and prior() share the same forward pass when
    they hit the same state (a leaf that later gets expanded).  Cache is cleared
    before each mcts_search() to stay consistent with network weights.
    """
    def __init__(self, network, device=DEVICE):
        self.network = network
        self.device  = device
        self._cache  = {}

    def clear_cache(self):
        self._cache = {}

    def _forward(self, state):
        key = str(state)
        if key not in self._cache:
            with torch.no_grad():
                logits, value = self.network(state_to_tensor(state, self.device))
            self._cache[key] = (logits.squeeze().cpu().numpy(), value.item())
        return self._cache[key]

    def evaluate(self, state):
        _, v = self._forward(state)
        cur  = state.current_player()
        ret  = [-v, -v]
        ret[cur] = v
        return ret

    def prior(self, state):
        legal = state.legal_actions()
        if not legal:
            return []
        logits, _ = self._forward(state)
        lg    = logits[legal] - logits[legal].max()
        probs = np.exp(lg)
        probs /= probs.sum()
        return list(zip(legal, probs.tolist()))


def make_bot(game, evaluator, num_simulations):
    return mcts_lib.MCTSBot(
        game,
        uct_c=1.4,
        max_simulations=num_simulations,
        evaluator=evaluator,
        dirichlet_noise=(0.25, 0.3),
    )


# ── Self-play ──────────────────────────────────────────────────────────────────

def _solved_adjust_counts(children, legal, counts):
    """MCTS-Solver can prove a position solved (root.outcome set) after
    exploring just ONE winning child, and the search then correctly stops
    early to save compute -- but that leaves raw visit counts almost
    meaningless (most children got 0-1 visits). If ANY legal child is a
    PROVEN WIN for the mover, replace `counts` with a distribution over ONLY
    the proven-winning actions (never fall back to a lower-value move once a
    forced win is found). Else if some (not all) children are PROVEN LOSSES,
    zero those out so a losing move is never preferred over an uncertain or
    better one. Otherwise return `counts` unchanged (the common, unsolved
    case, no different from before this fix).
    """
    by_action = {c.action: c for c in children}
    win_mask  = np.zeros(len(legal), dtype=bool)
    loss_mask = np.zeros(len(legal), dtype=bool)
    for i, a in enumerate(legal):
        c = by_action.get(a)
        if c is not None and c.outcome is not None:
            if c.outcome[c.player] == 1.0:
                win_mask[i] = True
            elif c.outcome[c.player] == -1.0:
                loss_mask[i] = True
    if win_mask.any():
        out = np.zeros_like(counts)
        out[win_mask] = 1.0
        return out
    if loss_mask.any() and not loss_mask.all():
        out = counts.copy()
        out[loss_mask] = 0.0
        if out.sum() > 0:
            return out
    return counts


def self_play_game(game, bot, temp_threshold=30, evaluator=None,
                   value_target_lambda=0.5):
    """
    MCTS self-play. Temperature annealing applies to ACTION SELECTION only
    (1.0 before temp_threshold, near-greedy 0.05 after); the policy TARGET is
    always the raw visit distribution (the unbiased improvement-operator
    output). Value targets are lam*z + (1-lam)*root_MCTS_value — the final
    outcome z alone is near coin-flip noise for balanced games.
    Returns list of (obs_flat, policy_vec, value_target) per move.
    """
    state    = game.new_initial_state()
    history  = []
    move_num = 0

    while not state.is_terminal():
        player      = state.current_player()
        obs         = state.observation_tensor(player)
        temperature = 1.0 if move_num < temp_threshold else 0.05

        # Flush per-search cache so evaluate() and prior() share computation
        # for the same leaf state within this search, without cross-search staleness.
        if evaluator is not None:
            evaluator.clear_cache()
        root      = bot.mcts_search(state)
        legal     = state.legal_actions()
        visit_map = {ch.action: ch.explore_count for ch in root.children}
        counts    = np.array([visit_map.get(a, 0) for a in legal], dtype=np.float32)
        counts    = _solved_adjust_counts(root.children, legal, counts)

        if temperature < 1e-2 or counts.sum() == 0:
            probs          = np.zeros(len(legal), dtype=np.float32)
            probs[counts.argmax()] = 1.0
        else:
            log_ct = np.log(counts + 1e-10) / temperature
            log_ct -= log_ct.max()
            ct     = np.exp(log_ct)
            probs  = ct / ct.sum()

        # Policy target: raw visit distribution (NOT temperature-sharpened).
        visit_dist = (counts / counts.sum() if counts.sum() > 0
                      else np.full(len(legal), 1.0 / len(legal), dtype=np.float32))
        policy_vec = np.zeros(_NUM_ACTIONS, dtype=np.float32)
        for a, p in zip(legal, visit_dist):
            policy_vec[a] = p
        root_q = root.total_reward / max(root.explore_count, 1)  # mover's view

        action = np.random.choice(legal, p=probs)
        history.append((player, list(obs), policy_vec, root_q))
        state.apply_action(action)
        move_num += 1

    returns = state.returns()
    lam = value_target_lambda
    return [(obs, pol, lam * returns[pl] + (1.0 - lam) * q)
            for pl, obs, pol, q in history]


# ── Evaluation ─────────────────────────────────────────────────────────────────

def greedy_action(network, state, device=DEVICE, temperature=0.0):
    with torch.no_grad():
        logits, _ = network(state_to_tensor(state, device))
    logits = logits.squeeze().cpu().numpy()
    legal  = state.legal_actions()
    if temperature == 0.0:
        return legal[int(logits[legal].argmax())]
    lg = logits[legal] - logits[legal].max()
    probs = np.exp(lg / temperature)
    probs /= probs.sum()
    return int(np.random.choice(legal, p=probs))


def eval_vs_random(game, network, num_games=100, device=DEVICE):
    """Returns (wins, draws, losses) for network (greedy, P0) vs random (P1)."""
    wins, draws, losses = 0, 0, 0
    for _ in range(num_games):
        state = game.new_initial_state()
        while not state.is_terminal():
            if state.current_player() == 0:
                action = greedy_action(network, state, device)
            else:
                action = random.choice(state.legal_actions())
            state.apply_action(action)
        r = state.returns()[0]
        if r > 0:    wins   += 1
        elif r < 0:  losses += 1
        else:        draws  += 1
    return wins, draws, losses


def eval_vs_snapshot(game, network, snapshot, num_games=100, device=DEVICE):
    """Returns (wins, draws, losses); alternates colors and samples with temperature=1."""
    wins, draws, losses = 0, 0, 0
    for i in range(num_games):
        state = game.new_initial_state()
        net_player = i % 2  # alternate: net plays P0 in even games, P1 in odd
        while not state.is_terminal():
            net    = network if state.current_player() == net_player else snapshot
            action = greedy_action(net, state, device, temperature=1.0)
            state.apply_action(action)
        r = state.returns()[net_player]
        if r > 0:    wins   += 1
        elif r < 0:  losses += 1
        else:        draws  += 1
    return wins, draws, losses


def update_elo(elo, opp_elo, win_rate, k=32):
    expected = 1.0 / (1.0 + 10 ** ((opp_elo - elo) / 400.0))
    return elo + k * (win_rate - expected)


# ── Benchmark-pool evaluation: running-Elo tournaments (no fixed anchor) ─────────
# Every net (random, the untrained '0' net, and each checkpoint) is a tournament
# player entering at START_ELO and updated incrementally with the Elo K-factor.
# Evaluation is run in independent TRACKS: policy-only (0 sims) and one per MCTS
# budget. Each track keeps its own Elo table and its own head-to-head matrix, so
# `mcts50` ratings never mix with `policy` ratings.

import math
import itertools
from collections import defaultdict


def play_match(net_a, net_b, game, n_games, device, temp=1.0):
    """net_a vs net_b over n_games, alternating colours. net == None ==> random.
    Policy-only (no MCTS). Returns (wins_a, draws, wins_b) from net_a's view.
    Used by the quick eval as a lightweight progress pulse (does not touch Elo)."""
    wa = dd = wb = 0
    for i in range(n_games):
        state = game.new_initial_state()
        a_player = i % 2
        while not state.is_terminal():
            net = net_a if state.current_player() == a_player else net_b
            if net is None:
                action = random.choice(state.legal_actions())
            else:
                action = greedy_action(net, state, device, temperature=temp)
            state.apply_action(action)
        r = state.returns()[a_player]
        if r > 0:   wa += 1
        elif r < 0: wb += 1
        else:       dd += 1
    return wa, dd, wb


def _mcts_move(bot, state):
    """Most-visited action from an MCTS search (root Dirichlet noise gives the
    per-game variety that keeps a round-robin from replaying identical games)."""
    root   = bot.mcts_search(state)
    legal  = state.legal_actions()
    visits = {c.action: c.explore_count for c in root.children}
    counts = np.array([visits.get(a, 0) for a in legal], dtype=np.float32)
    return legal[int(counts.argmax())]


def run_tournament(players, elos, wdl, game, device,
                   games_per_pair=10, k=32.0, temp=1.0, sims=0, opening_plies=0,
                   focus_label=None, refresh_pairs=0):
    """One full round-robin for a single track. Moves are chosen by policy-only
    (sims == 0: greedy) or by MCTS with `sims` simulations (most-visited move, no
    root Dirichlet noise). The first `opening_plies` moves of each game are random
    (both sides) only to vary games; strength is then measured noise-free. Every unordered pair plays `games_per_pair` games, half
    per colour, all shuffled into random order; Elo is updated after each game.
    `elos` must already hold a rating for every label. Mutates `elos` and `wdl`."""
    net  = {p['label']: p['net'] for p in players}
    bots = {}
    if sims > 0:
        bs = max(1, min(sims, 16))
        for p in players:
            if p['net'] is not None:
                bots[p['label']] = make_batched_bot(game, p['net'], device, sims, bs,
                                                    dirichlet=None)

    def pick(label, state):
        if net[label] is None:
            return random.choice(state.legal_actions())
        if sims <= 0:
            return greedy_action(net[label], state, device, temperature=0.0)
        return _mcts_move(bots[label], state)

    pairs = list(itertools.combinations(net.keys(), 2))
    if focus_label is not None:
        # Linear-cost mode: every pair involving the newest model, plus a few
        # random old-vs-old pairs so earlier ratings keep mixing. Full
        # round-robin cost grows O(pool^2) per deep eval; this stays O(pool).
        focus  = [p for p in pairs if focus_label in p]
        others = [p for p in pairs if focus_label not in p]
        random.shuffle(others)
        pairs = focus + others[:refresh_pairs]
    schedule = []
    for a, b in pairs:
        half = games_per_pair // 2
        schedule += [(a, b)] * half
        schedule += [(b, a)] * half
        schedule += [(a, b)] * (games_per_pair - 2 * half)   # odd remainder
    random.shuffle(schedule)

    for p0, p1 in schedule:
        state = game.new_initial_state()
        ply = 0
        while not state.is_terminal():
            if ply < opening_plies:
                action = random.choice(state.legal_actions())  # random opening → variety
            else:
                lab = p0 if state.current_player() == 0 else p1
                action = pick(lab, state)
            state.apply_action(action)
            ply += 1
        r  = state.returns()[0]
        s0 = 1.0 if r > 0 else (0.0 if r < 0 else 0.5)
        e0 = 1.0 / (1.0 + 10 ** ((elos[p1] - elos[p0]) / 400.0))
        elos[p0] += k * (s0 - e0)
        elos[p1] += k * ((1.0 - s0) - (1.0 - e0))
        cell = wdl[(p0, p1)]
        if r > 0:   cell[0] += 1
        elif r < 0: cell[2] += 1
        else:       cell[1] += 1


In [ ]:
import math
import os


# ── Batched-leaf MCTS (KataGo's core speedup) ───────────────────────────────────
# Vanilla MCTS evaluates one leaf per NN forward pass. On CPU/GPU that wastes
# almost all throughput: a forward pass on a batch of 16 states costs barely more
# than a batch of 1. This bot collects a *wave* of `batch_size` leaves using
# **virtual loss**, evaluates them all in ONE forward pass, then backs them all
# up — typically 10-30× more simulations per second.
#
# Virtual loss: while a leaf is "in flight" (selected but not yet evaluated) we
# temporarily pretend the path lost. This pushes subsequent selections in the
# same wave down *different* branches so we gather a diverse batch instead of the
# same leaf 16 times. The virtual loss is removed when the real value is backed up.

class _BNode:
    """Lightweight search node with a virtual-loss counter."""
    __slots__ = ['action', 'player', 'prior', 'explore_count',
                 'total_reward', 'vloss', 'outcome', 'children']

    def __init__(self, action, player, prior):
        self.action        = action
        self.player        = player
        self.prior         = prior
        self.explore_count = 0
        self.total_reward  = 0.0
        self.vloss         = 0          # in-flight virtual visits
        self.outcome       = None
        self.children      = []


class BatchedMCTSBot:
    """
    Drop-in replacement for mcts_lib.MCTSBot exposing mcts_search(state) -> root.
    `root.children` are _BNode with .action and .explore_count, matching the
    interface self_play_game() consumes.

    Weights are read live from `network`, so in-place updates take effect
    immediately (no per-search cache needed — the batch IS the speedup).
    """
    def __init__(self, game, network, device, max_simulations,
                 batch_size=16, uct_c=1.4, dirichlet=(0.25, 0.3),
                 solve=True, random_state=None):
        self.game            = game
        self.network         = network
        self.device          = device
        self.max_simulations = max_simulations
        self.batch_size      = batch_size
        self.uct_c           = uct_c
        self.dirichlet       = dirichlet
        # Forced playouts are a self-play device tied to root noise; evaluation
        # bots (dirichlet=None) never force and never prune.
        self.k_forced        = 2.0 if dirichlet is not None else 0.0
        self.solve           = solve
        self.max_utility     = game.max_utility()
        self._rng            = random_state or np.random.RandomState()

    # ── NN: evaluate a list of states in one forward pass ──────────────────────
    def _eval_batch(self, states):
        obs_list = [s.observation_tensor(s.current_player()) for s in states]
        x = batch_to_tensor(obs_list, self.device)          # (B, 9, 6, 6)
        with torch.no_grad():
            logits, values = self.network(x)
        return (logits.cpu().numpy(),                       # (B, _NUM_ACTIONS)
                values.squeeze(-1).cpu().numpy())           # (B,)

    # ── Expand a node: turn NN policy logits into prior-weighted children ──────
    def _expand(self, node, cur_player, legal, logits, add_noise):
        lg = logits[legal] - logits[legal].max()
        pr = np.exp(lg)
        pr /= pr.sum()
        if add_noise and self.dirichlet is not None:
            eps, alpha = self.dirichlet
            noise = self._rng.dirichlet([alpha] * len(legal))
            pr = (1.0 - eps) * pr + eps * noise
        children = [_BNode(a, cur_player, float(p)) for a, p in zip(legal, pr)]
        self._rng.shuffle(children)                         # reduce move-order bias
        node.children = children

    # ── PUCT with virtual loss folded in ───────────────────────────────────────
    def _puct(self, child, parent_n_eff):
        if child.outcome is not None:
            return child.outcome[child.player]
        ec = child.explore_count + child.vloss             # effective visits
        q  = (child.total_reward - child.vloss) / ec if ec > 0 else 0.0
        return q + self.uct_c * child.prior * math.sqrt(parent_n_eff) / (ec + 1)

    # ── Descend from root to an unexpanded/terminal leaf, applying virtual loss ─
    def _select_leaf(self, root, root_state):
        path  = [root]
        root.vloss += 1
        state = root_state.clone()
        node  = root
        while node.children and not state.is_terminal():
            parent_n_eff = node.explore_count + node.vloss
            best = None
            if node is root and self.k_forced > 0.0:
                # KataGo forced playouts: every noised root child must receive
                # at least sqrt(k * P(c) * N) visits, so noise-seeded moves are
                # actually explored (target pruning later removes what they
                # didn't earn). Virtual losses count as in-flight visits.
                total = sum(c.explore_count + c.vloss for c in node.children)
                if total > 0:
                    deficit, forced = 0.0, None
                    for c in node.children:
                        need = (math.sqrt(self.k_forced * c.prior * total)
                                - (c.explore_count + c.vloss))
                        if need > deficit:
                            deficit, forced = need, c
                    best = forced
            if best is None:
                best = max(node.children,
                           key=lambda c: self._puct(c, parent_n_eff))
            state.apply_action(best.action)
            best.vloss += 1
            path.append(best)
            node = best
        return path, state, node

    # ── Back up a network value estimate (non-terminal leaf) ───────────────────
    def _backup_value(self, path, leaf_cur, value):
        for node in reversed(path):
            node.vloss         -= 1
            node.explore_count += 1
            node.total_reward  += value if node.player == leaf_cur else -value

    # ── Back up a terminal result, propagating solved outcomes (MCTS-Solver) ───
    def _backup_terminal(self, path, returns):
        path[-1].outcome = returns
        solved = self.solve
        for node in reversed(path):
            node.vloss         -= 1
            node.explore_count += 1
            node.total_reward  += returns[node.player]
            if solved and node.children:
                player = node.children[0].player
                best, all_solved = None, True
                for ch in node.children:
                    if ch.outcome is None:
                        all_solved = False
                    elif best is None or ch.outcome[player] > best.outcome[player]:
                        best = ch
                if best is not None and (all_solved or
                                         best.outcome[player] == self.max_utility):
                    node.outcome = best.outcome
                else:
                    solved = False

    # ── Main search: waves of batched-leaf evaluation ──────────────────────────
    def mcts_search(self, state):
        root = _BNode(None, state.current_player(), 1.0)

        # Seed: expand root (with Dirichlet noise) and back up its own value.
        logits, values = self._eval_batch([state])
        self._expand(root, state.current_player(), state.legal_actions(),
                     logits[0], add_noise=True)
        root.explore_count = 1
        root.total_reward += float(values[0])

        while root.explore_count < self.max_simulations:
            if root.outcome is not None:
                break
            wave = min(self.batch_size, self.max_simulations - root.explore_count)

            # 1. Collect a diverse wave of leaves via virtual loss.
            pending = []
            for _ in range(wave):
                if root.outcome is not None:
                    break
                path, st, leaf = self._select_leaf(root, state)
                pending.append((path, leaf, st))

            # 2. Evaluate every unique non-terminal leaf in ONE forward pass.
            to_eval = {}
            for path, leaf, st in pending:
                if not st.is_terminal():
                    to_eval[id(leaf)] = (leaf, st)

            results = {}
            if to_eval:
                states = [s for (_, s) in to_eval.values()]
                lg, vv = self._eval_batch(states)
                for (k, (leaf, st)), logit_row, val in zip(to_eval.items(), lg, vv):
                    cur = st.current_player()
                    self._expand(leaf, cur, st.legal_actions(), logit_row, False)
                    results[k] = (cur, float(val))

            # 3. Back up all leaves (terminals resolved exactly; others via NN).
            for path, leaf, st in pending:
                if st.is_terminal():
                    self._backup_terminal(path, st.returns())
                else:
                    cur, val = results[id(leaf)]
                    self._backup_value(path, cur, val)

        return root


def make_batched_bot(game, network, device, num_simulations,
                     batch_size=16, uct_c=1.4, dirichlet=(0.25, 0.3)):
    return BatchedMCTSBot(game, network, device, num_simulations,
                          batch_size=batch_size, uct_c=uct_c, dirichlet=dirichlet)


def _pruned_visit_counts(root, uct_c, k_forced):
    """KataGo policy-target pruning (Wu 2020): subtract root-child visits that
    exist only because of forced playouts / Dirichlet noise, keeping the visits
    each move earned on merit — those its PUCT score justifies against the
    most-visited child. Children left with <=1 visit are zeroed. The
    most-visited child always keeps every visit. Returns {action: count}."""
    kids = root.children
    best = max(kids, key=lambda c: c.explore_count)
    total = sum(c.explore_count for c in kids)
    if total <= 0 or best.explore_count <= 0 or k_forced <= 0:
        return {c.action: c.explore_count for c in kids}
    sqrt_total = math.sqrt(total)
    q_best = best.total_reward / best.explore_count
    puct_best = (q_best + uct_c * best.prior * sqrt_total
                 / (1 + best.explore_count))
    out = {}
    for c in kids:
        n = c.explore_count
        if c is best or n == 0:
            out[c.action] = n
            continue
        d = puct_best - c.total_reward / n
        if d <= 0:
            out[c.action] = n              # beats best's PUCT: fully earned
            continue
        # Smallest count at which this child's PUCT still stays below the
        # best child's — visits above that are candidates for removal, capped
        # by the forced-playout allotment sqrt(k * P * N).
        n_min    = uct_c * c.prior * sqrt_total / d - 1.0
        n_forced = math.sqrt(k_forced * c.prior * total)
        keep = min(n, max(int(math.ceil(n_min)), n - int(n_forced), 0))
        out[c.action] = 0 if keep <= 1 else keep
    return out


# ── Parallel self-play: many games, one forward pass ────────────────────────────
# BatchedMCTSBot batches leaves within ONE search (≤ its batch_size). The next
# multiplier is batching ACROSS games: run N games concurrently, collect every
# game's leaf wave, and evaluate them all in a single NN forward pass. The joint
# batch is ≈ n_parallel × wave_per_game (e.g. 8×8 = 64), which uses the hardware
# far better than 16 — and the advantage grows with network size.

def _nn_eval_states(network, device, states):
    """Evaluate a list of states in one forward pass → (logits (B,A), values (B,))."""
    obs_list = [s.observation_tensor(s.current_player()) for s in states]
    x = batch_to_tensor(obs_list, device)
    with torch.no_grad():
        logits, values = network(x)
    return logits.cpu().numpy(), values.squeeze(-1).cpu().numpy()


class ParallelSelfPlay:
    """Interleaves the MCTS searches of n_parallel self-play games so their leaf
    waves share one NN forward pass. Tree operations (selection with virtual
    loss, expansion, backup, MCTS-Solver) are reused from BatchedMCTSBot; this
    class only orchestrates the games and the joint evaluation.

    Weights are read live from `network`, so games in flight pick up training
    updates immediately (standard asynchronous self-play behaviour)."""

    def __init__(self, game, network, device, n_parallel=8, wave_per_game=8,
                 fast_sims=20, full_sims=100, fast_prob=0.75, temp_threshold=30,
                 uct_c=1.4, dirichlet=(0.25, 0.3), value_target_lambda=0.5,
                 pool_prob=0.0, checkpoint_dir=None, channels=None, num_blocks=None):
        self.game = game
        self.network = network
        self.device = device
        self.n_parallel = n_parallel
        self.wave = wave_per_game
        self.fast_sims = fast_sims
        self.full_sims = full_sims
        self.fast_prob = fast_prob
        self.temp_threshold = temp_threshold
        # Opponent diversity (see _resolve_pool_moves): some fraction of games
        # have one side played by a frozen historical checkpoint instead of
        # mirror-matching the live net against itself.
        self.pool_prob = pool_prob
        self.checkpoint_dir = checkpoint_dir
        self.channels = channels
        self.num_blocks = num_blocks
        self._pool_nets = {}          # benchmark label -> frozen CPU BoopNet
        # Value target = lam*z + (1-lam)*root_MCTS_value. Pure z is near
        # coin-flip noise when both sides play well; the root value estimate is
        # far lower-variance (KataGo-style target mixing).
        self.vt_lambda = value_target_lambda
        # One bot per slot supplies the tree ops; searches are driven externally.
        self.bots = [BatchedMCTSBot(game, network, device, full_sims,
                                    batch_size=wave_per_game, uct_c=uct_c,
                                    dirichlet=dirichlet)
                     for _ in range(n_parallel)]
        self.slots = [self._new_game() for _ in range(n_parallel)]

    def _load_pool_net(self, label):
        net = self._pool_nets.get(label)
        if net is None:
            path = os.path.join(self.checkpoint_dir, f'bench_{label}.pt')
            net = BoopNet(self.channels, self.num_blocks)
            net.load_state_dict(torch.load(path, map_location='cpu',
                                           weights_only=True))
            net.eval()
            self._pool_nets[label] = net
        return net

    def _new_game(self):
        # Playout cap randomization is chosen per game, as in serial self-play.
        sims = self.fast_sims if random.random() < self.fast_prob else self.full_sims
        slot = {'state': self.game.new_initial_state(), 'hist': [],
                'move': 0, 'sims': sims, 'root': None, 'pool': None}
        if self.pool_prob > 0 and self.checkpoint_dir:
            try:
                labels = [f[6:-3] for f in os.listdir(self.checkpoint_dir)
                          if f.startswith('bench_') and f.endswith('.pt')]
            except OSError:
                labels = []
            if labels and random.random() < self.pool_prob:
                label = random.choice(labels)
                slot['pool'] = {'label': label, 'side': random.randint(0, 1),
                                'net': self._load_pool_net(label)}
        return slot

    def _resolve_pool_moves(self):
        """Resolve any pending frozen-opponent moves with a direct forward
        pass + greedy (argmax) choice -- no search tree, since these moves are
        never used as training targets. Returns finished episodes (a pool move
        can end the game), matching _step()'s return contract."""
        done = []
        for i, s in enumerate(self.slots):
            pool = s['pool']
            if pool is None:
                continue
            state = s['state']
            if state.current_player() != pool['side']:
                continue
            logits, _ = _nn_eval_states(pool['net'], 'cpu', [state])
            legal = state.legal_actions()
            action = legal[int(np.argmax(logits[0][legal]))]
            state.apply_action(action)
            s['move'] += 1
            if state.is_terminal():
                returns = state.returns()
                lam = self.vt_lambda
                done.append([(obs, pol, lam * returns[pl] + (1.0 - lam) * q)
                             for pl, obs, pol, q in s['hist']])
                self.slots[i] = self._new_game()
        return done

    def episodes(self):
        """Infinite generator of finished episodes: lists of (obs, policy, return)."""
        while True:
            for data in self._step():
                yield data

    def _step(self):
        """Advance every game by one leaf wave (one joint forward pass);
        returns any episodes that finished this step."""
        done = self._resolve_pool_moves()
        pending = []                               # (slot_idx, path, leaf, state)
        for i, s in enumerate(self.slots):
            pool = s['pool']
            if pool is not None and s['state'].current_player() == pool['side']:
                continue   # next move belongs to the frozen opponent
            if s['root'] is None:
                s['root'] = _BNode(None, s['state'].current_player(), 1.0)
            root = s['root']
            if root.outcome is not None:
                continue                           # solved — finalized below
            # An unexpanded root must be evaluated alone (wave of 1, matching the
            # serial bot's root seeding); afterwards collect full waves under
            # virtual loss.
            wave = 1 if not root.children else min(self.wave,
                                                   s['sims'] - root.explore_count)
            for _ in range(max(wave, 0)):
                if root.outcome is not None:
                    break
                path, wstate, leaf = self.bots[i]._select_leaf(root, s['state'])
                pending.append((i, path, leaf, wstate))

        # ONE forward pass for every unique non-terminal leaf across all games.
        to_eval = {}
        for i, path, leaf, wstate in pending:
            if not wstate.is_terminal() and id(leaf) not in to_eval:
                to_eval[id(leaf)] = (i, leaf, wstate)
        results = {}
        if to_eval:
            entries = list(to_eval.values())
            logits, values = _nn_eval_states(self.network, self.device,
                                             [w for _, _, w in entries])
            for (i, leaf, wstate), lg, val in zip(entries, logits, values):
                self.bots[i]._expand(leaf, wstate.current_player(),
                                     wstate.legal_actions(), lg,
                                     add_noise=(leaf is self.slots[i]['root']))
                results[id(leaf)] = (wstate.current_player(), float(val))
        for i, path, leaf, wstate in pending:
            if wstate.is_terminal():
                self.bots[i]._backup_terminal(path, wstate.returns())
            else:
                cur, val = results[id(leaf)]
                self.bots[i]._backup_value(path, cur, val)

        # Finalize finished searches; emit any completed episodes (keeping
        # any already emitted by _resolve_pool_moves above).
        for i, s in enumerate(self.slots):
            root = s['root']
            if root is None or root.explore_count == 0:
                continue
            if root.explore_count < s['sims'] and root.outcome is None:
                continue
            self._play_move(i)
            if s['state'].is_terminal():
                returns = s['state'].returns()
                lam = self.vt_lambda
                done.append([(obs, pol, lam * returns[pl] + (1.0 - lam) * q)
                             for pl, obs, pol, q in s['hist']])
                self.slots[i] = self._new_game()
        return done

    def _play_move(self, i):
        """Turn a finished search into a move + training sample (same temperature
        scheme as self_play_game)."""
        s      = self.slots[i]
        state  = s['state']
        player = state.current_player()
        obs    = state.observation_tensor(player)
        temperature = 1.0 if s['move'] < self.temp_threshold else 0.05

        legal     = state.legal_actions()
        visit_map = {ch.action: ch.explore_count for ch in s['root'].children}
        counts    = np.array([visit_map.get(a, 0) for a in legal],
                             dtype=np.float32)
        counts    = _solved_adjust_counts(s['root'].children, legal, counts)
        if temperature < 1e-2 or counts.sum() == 0:
            probs = np.zeros(len(legal), dtype=np.float32)
            probs[counts.argmax()] = 1.0
        else:
            log_ct  = np.log(counts + 1e-10) / temperature
            log_ct -= log_ct.max()
            ct      = np.exp(log_ct)
            probs   = ct / ct.sum()

        # Policy TARGET: PRUNED visit distribution (KataGo) — visits that exist
        # only because of forced playouts / Dirichlet noise are removed, so the
        # policy stops learning to imitate exploration noise. The move itself
        # is still selected from the RAW counts above.
        bot     = self.bots[i]
        pvis    = _pruned_visit_counts(s['root'], bot.uct_c, bot.k_forced)
        pcounts = np.array([pvis.get(a, 0) for a in legal], dtype=np.float32)
        pcounts = _solved_adjust_counts(s['root'].children, legal, pcounts)
        if pcounts.sum() <= 0:
            pcounts = counts
        visit_dist = (pcounts / pcounts.sum() if pcounts.sum() > 0
                      else np.full(len(legal), 1.0 / len(legal), dtype=np.float32))
        policy_vec = np.zeros(_NUM_ACTIONS, dtype=np.float32)
        for a, p in zip(legal, visit_dist):
            policy_vec[a] = p
        root   = s['root']
        root_q = root.total_reward / max(root.explore_count, 1)  # mover's view

        s['hist'].append((player, list(obs), policy_vec, root_q))
        state.apply_action(int(np.random.choice(legal, p=probs)))
        s['move'] += 1
        s['root']  = None


In [ ]:
# ── Multiprocessing self-play + central GPU inference server ─────────────────
# The DirectML lesson from this project: the GPU loses on small batches (per-op
# dispatch + transfer latency) and wins on big ones. So: N CPU worker processes
# do ALL game/tree work (pure Python — processes sidestep the GIL) and ship
# their NN requests to ONE server thread here, which batches requests from all
# workers into single large forward passes on the GPU. Small-batch inference
# (eval games, tournaments) runs on CPU replicas elsewhere.
#
# Windows/Jupyter constraint: spawned processes cannot pickle notebook-cell
# functions, so the worker code is written to a real module file first.
import pathlib

_WORKER_SRC = r'''# AUTO-GENERATED by boop_kataboop_training.ipynb — do not edit by hand.
# Self-play worker module: game engine + MCTS tree ops + worker loop.
# Exists as a real file because Windows multiprocessing (spawn) cannot
# pickle functions defined inside notebook cells.

# Boop board game -- inlined so Colab works without the custom package.
# Faithful rules: graduation conserves pieces (kittens BECOME cats), and a
# player whose pool is empty must graduate a kitten of their choice rather than
# being stuck (which previously caused spurious draws).
# Perf: win checks, promotion, legal moves and the observer are vectorized over
# a precomputed 3-in-a-row line table, and clone() is a hand-rolled field copy
# instead of deepcopy — MCTS clones a state once per simulation, so these are
# the hottest paths in the whole pipeline.

import numpy as np
from open_spiel.python.observation import IIGObserverForPublicInfoGame
import pyspiel

_NUM_PLAYERS = 2
_ROWS = 6
_COLS = 6
_NUM_CELLS = _ROWS * _COLS
_NUM_PIECE_TYPES = 2
_NUM_PIECES = 8                              # each player has exactly 8 pieces
_GRADUATE_OFFSET = _NUM_PIECE_TYPES * _NUM_CELLS  # 72: graduation actions start here
_NUM_ACTIONS = _GRADUATE_OFFSET + _NUM_CELLS      # 108 = 72 placement + 36 graduate
_MAX_KITTENS = 8
_MAX_CATS = 8                               # all 8 pieces can become cats
_MAX_GAME_LENGTH = 500

_EMPTY = 0
_P0_KITTEN = 1
_P0_CAT = 2
_P1_KITTEN = 3
_P1_CAT = 4

_KITTEN_VAL = [_P0_KITTEN, _P1_KITTEN]
_CAT_VAL = [_P0_CAT, _P1_CAT]
_PIECE_VALS = [[_P0_KITTEN, _P0_CAT], [_P1_KITTEN, _P1_CAT]]

# Every 3-in-a-row line on the board (horizontal, vertical, both diagonals) as
# indices into the flattened 36-cell board: shape (80, 3). Win detection and
# promotion become single vectorized gathers instead of triple Python loops.
_LINE_IDX = np.array(
    [[(r + k * dr) * _COLS + (c + k * dc) for k in range(3)]
     for r in range(_ROWS) for c in range(_COLS)
     for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1))
     if 0 <= r + 2 * dr < _ROWS and 0 <= c + 2 * dc < _COLS],
    dtype=np.int64)

_GAME_TYPE = pyspiel.GameType(
    short_name='python_boop',
    long_name='Python Boop',
    dynamics=pyspiel.GameType.Dynamics.SEQUENTIAL,
    chance_mode=pyspiel.GameType.ChanceMode.DETERMINISTIC,
    information=pyspiel.GameType.Information.PERFECT_INFORMATION,
    utility=pyspiel.GameType.Utility.ZERO_SUM,
    reward_model=pyspiel.GameType.RewardModel.TERMINAL,
    max_num_players=_NUM_PLAYERS,
    min_num_players=_NUM_PLAYERS,
    provides_information_state_string=True,
    provides_information_state_tensor=False,
    provides_observation_string=True,
    provides_observation_tensor=True,
    parameter_specification={})

_GAME_INFO = pyspiel.GameInfo(
    num_distinct_actions=_NUM_ACTIONS,
    max_chance_outcomes=0,
    num_players=_NUM_PLAYERS,
    min_utility=-1.0,
    max_utility=1.0,
    utility_sum=0.0,
    max_game_length=_MAX_GAME_LENGTH)


class BoopGame(pyspiel.Game):
  def __init__(self, params=None):
    super().__init__(_GAME_TYPE, _GAME_INFO, params or dict())

  def new_initial_state(self):
    return BoopState(self)

  def make_py_observer(self, iig_obs_type=None, params=None):
    if ((iig_obs_type is None) or
        (iig_obs_type.public_info and not iig_obs_type.perfect_recall)):
      return BoopObserver(params)
    return IIGObserverForPublicInfoGame(iig_obs_type, params)


class BoopState(pyspiel.State):
  def __init__(self, game):
    super().__init__(game)
    self._game_ref = game
    self._cur_player = 0
    self._is_terminal = False
    self._winner = None
    self._move_count = 0
    self.board = np.zeros((_ROWS, _COLS), dtype=np.int8)
    self._hand = [[_MAX_KITTENS, 0], [_MAX_KITTENS, 0]]

  def clone(self):
    # Fast clone: MCTS calls this once per simulation. Copying the handful of
    # fields directly is ~10x cheaper than the default deepcopy-based clone.
    cp = BoopState(self._game_ref)
    cp._cur_player  = self._cur_player
    cp._is_terminal = self._is_terminal
    cp._winner      = self._winner
    cp._move_count  = self._move_count
    cp.board        = self.board.copy()
    cp._hand        = [self._hand[0][:], self._hand[1][:]]
    return cp

  def current_player(self):
    return pyspiel.PlayerId.TERMINAL if self._is_terminal else self._cur_player

  def _legal_actions(self, player):
    # Forced-graduation rule: if the pool is empty, the player must graduate one
    # of their kittens on the board (returning it to the pool as a cat) instead
    # of placing. Graduation actions are _GRADUATE_OFFSET + cell.
    flat = self.board.reshape(-1)
    hk, hc = self._hand[player]
    if hk == 0 and hc == 0:
      kittens = np.flatnonzero(flat == _KITTEN_VAL[player])
      return (_GRADUATE_OFFSET + kittens).tolist()
    empty = np.flatnonzero(flat == _EMPTY)
    if hk > 0 and hc > 0:
      return np.concatenate([empty, _NUM_CELLS + empty]).tolist()
    if hk > 0:
      return empty.tolist()
    return (_NUM_CELLS + empty).tolist()

  def _apply_action(self, action):
    p = self._cur_player
    if action >= _GRADUATE_OFFSET:
      # Forced graduation: a kitten on the board becomes a cat in the pool.
      cell = action - _GRADUATE_OFFSET
      r, c = cell // _COLS, cell % _COLS
      self.board[r, c] = _EMPTY
      self._hand[p][1] += 1
      self._move_count += 1
      self._post_move(p)        # no piece placed → no boop
      return
    piece_type = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    self._hand[p][piece_type] -= 1
    self.board[r, c] = _PIECE_VALS[p][piece_type]
    self._boop(r, c, is_cat=(piece_type == 1))
    self._move_count += 1
    self._post_move(p)

  def _post_move(self, p):
    """Shared post-move resolution: win checks, promotion, turn handoff."""
    if self._move_count >= _MAX_GAME_LENGTH:
      self._is_terminal = True
      return
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._promote_kittens(p)
    self._promote_kittens(1 - p)
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._cur_player = 1 - p
    # With forced graduation a player is never permanently stuck: an empty pool
    # forces a graduation, and a board of eight cats is already a win. The guard
    # below is defensive only and should not trigger in normal play.
    if not self._legal_actions(self._cur_player):
      self._is_terminal = True

  def _action_to_string(self, player, action):
    if action >= _GRADUATE_OFFSET:
      cell = action - _GRADUATE_OFFSET
      r, c = cell // _COLS, cell % _COLS
      return f'p{player}:graduate@({r},{c})'
    pt = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    piece = 'cat' if pt else 'kitten'
    return f'p{player}:{piece}@({r},{c})'

  def is_terminal(self):
    return self._is_terminal

  def returns(self):
    if self._winner == 0:
      return [1.0, -1.0]
    if self._winner == 1:
      return [-1.0, 1.0]
    return [0.0, 0.0]

  def __str__(self):
    syms = {
        _EMPTY: '.', _P0_KITTEN: 'k', _P0_CAT: 'K',
        _P1_KITTEN: 'o', _P1_CAT: 'O',
    }
    rows = [
        ''.join(syms[self.board[r, c]] for c in range(_COLS))
        for r in range(_ROWS)
    ]
    rows.append(
        f'P0: {self._hand[0][0]}k {self._hand[0][1]}K  '
        f'P1: {self._hand[1][0]}k {self._hand[1][1]}K  '
        f'move={self._move_count}')
    return '\n'.join(rows)

  def _boop(self, r, c, is_cat):
    board = self.board
    for dr in (-1, 0, 1):
      for dc in (-1, 0, 1):
        if dr == 0 and dc == 0:
          continue
        nr, nc = r + dr, c + dc
        if not (0 <= nr < _ROWS and 0 <= nc < _COLS):
          continue
        neighbor = board[nr, nc]
        if neighbor == _EMPTY:
          continue
        neighbor_is_cat = neighbor == _P0_CAT or neighbor == _P1_CAT
        if not is_cat and neighbor_is_cat:
          continue
        dest_r, dest_c = nr + dr, nc + dc
        owner = 0 if (neighbor == _P0_KITTEN or neighbor == _P0_CAT) else 1
        n_type = 1 if neighbor_is_cat else 0
        if not (0 <= dest_r < _ROWS and 0 <= dest_c < _COLS):
          board[nr, nc] = _EMPTY
          self._hand[owner][n_type] += 1
        elif board[dest_r, dest_c] == _EMPTY:
          board[dest_r, dest_c] = neighbor
          board[nr, nc] = _EMPTY

  def _promote_kittens(self, player):
    # Faithful rule (rulebook p.4): a line of 3 of the player's own pieces —
    # kittens AND/OR cats mixed — graduates. Every kitten in the line becomes
    # a cat; every cat in the line simply returns to the pool; either way all
    # three board cells clear and the pool gains 3 cats (pieces conserved).
    # A pure 3-cats-in-a-row is a WIN, checked before this runs, so it never
    # reaches here as a live case.
    kv, cv = _KITTEN_VAL[player], _CAT_VAL[player]
    flat = self.board.reshape(-1)
    while True:
      mine = (flat == kv) | (flat == cv)
      if int(mine.sum()) < 3:
        return
      full = mine[_LINE_IDX].all(axis=1)
      if not full.any():
        return
      # Resolve ONE qualifying line per pass, chosen UNIFORMLY AT RANDOM
      # among all lines that currently qualify (not the player's choice —
      # that would need a new action type, which isn't worth the added
      # action-space complexity). Clearing the chosen line's cells
      # invalidates any OVERLAPPING line, so it is not picked again this
      # call — matching "choose one, leave the rest" for a connected run of
      # 4+ (fig.4). An independent (non-overlapping) line elsewhere still
      # qualifies and is resolved on the loop's next pass.
      candidates = _LINE_IDX[full]
      line = candidates[np.random.randint(len(candidates))]
      flat[line] = _EMPTY
      self._hand[player][1] += 3

  def _check_win(self, player):
    flat = self.board.reshape(-1)
    cats = flat == _CAT_VAL[player]
    n = int(cats.sum())
    if n >= _NUM_PIECES:        # win condition 1: all eight pieces are cats
      return True
    if n < 3:
      return False
    # Win condition 2: three cats in a row (orthogonal or diagonal).
    return bool(cats[_LINE_IDX].all(axis=1).any())


class BoopObserver:
  def __init__(self, params):
    if params:
      raise ValueError(f'Observation parameters not supported; passed {params}')
    board_size = 5 * _ROWS * _COLS
    self.tensor = np.zeros(board_size + 4, np.float32)
    self.dict = {
        'observation': np.reshape(self.tensor[:board_size], (5, _ROWS, _COLS)),
        'hand': self.tensor[board_size:],
    }

  def set_from(self, state, player):
    self.tensor.fill(0)
    obs = self.dict['observation']
    hand = self.dict['hand']
    b = state.board
    opp = 1 - player
    obs[0][b == _EMPTY] = 1.0
    obs[1][b == _KITTEN_VAL[player]] = 1.0
    obs[2][b == _CAT_VAL[player]] = 1.0
    obs[3][b == _KITTEN_VAL[opp]] = 1.0
    obs[4][b == _CAT_VAL[opp]] = 1.0
    hand[0] = state._hand[player][0] / _MAX_KITTENS
    hand[1] = state._hand[player][1] / _MAX_CATS
    hand[2] = state._hand[opp][0] / _MAX_KITTENS
    hand[3] = state._hand[opp][1] / _MAX_CATS

  def string_from(self, state, player):
    del player
    return str(state)


try:
    pyspiel.register_game(_GAME_TYPE, BoopGame)
except Exception:
    pass  # already registered in this process


# ═══════════════════════════════════════════════════════════════════════════════
# MCTS tree ops — standalone copies of BatchedMCTSBot's semantics (virtual loss,
# PUCT, MCTS-Solver backup). Workers are pure CPU: no torch import; every NN
# evaluation is shipped to the main process's GPU inference server.
# ═══════════════════════════════════════════════════════════════════════════════
import math
import os
import random


class _WNode:
    __slots__ = ('action', 'player', 'prior', 'n', 'w', 'vloss',
                 'outcome', 'children')

    def __init__(self, action, player, prior):
        self.action = action
        self.player = player
        self.prior = prior
        self.n = 0
        self.w = 0.0
        self.vloss = 0
        self.outcome = None
        self.children = []


def _puct(ch, parent_n_eff, uct_c):
    if ch.outcome is not None:
        return ch.outcome[ch.player]
    ec = ch.n + ch.vloss
    q = (ch.w - ch.vloss) / ec if ec > 0 else 0.0
    return q + uct_c * ch.prior * math.sqrt(parent_n_eff) / (ec + 1)


def _select_leaf(root, root_state, uct_c, k_forced=0.0):
    path = [root]
    root.vloss += 1
    state = root_state.clone()
    node = root
    while node.children and not state.is_terminal():
        pne = node.n + node.vloss
        best = None
        if k_forced > 0.0 and node is root:
            # KataGo forced playouts: every noised root child must receive at
            # least sqrt(k * P(c) * N) visits (virtual losses count).
            total = sum(c.n + c.vloss for c in node.children)
            if total > 0:
                deficit, forced = 0.0, None
                for c in node.children:
                    need = math.sqrt(k_forced * c.prior * total) - (c.n + c.vloss)
                    if need > deficit:
                        deficit, forced = need, c
                best = forced
        if best is None:
            best = max(node.children, key=lambda c: _puct(c, pne, uct_c))
        state.apply_action(best.action)
        best.vloss += 1
        path.append(best)
        node = best
    return path, state, node


def _expand(node, cur_player, legal, logits_row, rng, dirichlet):
    lg = logits_row[legal] - logits_row[legal].max()
    pr = np.exp(lg)
    pr /= pr.sum()
    if dirichlet is not None:
        eps, alpha = dirichlet
        pr = (1.0 - eps) * pr + eps * rng.dirichlet([alpha] * len(legal))
    children = [_WNode(a, cur_player, float(p)) for a, p in zip(legal, pr)]
    rng.shuffle(children)
    node.children = children


def _backup_value(path, leaf_cur, value):
    for node in reversed(path):
        node.vloss -= 1
        node.n += 1
        node.w += value if node.player == leaf_cur else -value


def _backup_terminal(path, returns, max_utility=1.0):
    path[-1].outcome = returns
    solved = True
    for node in reversed(path):
        node.vloss -= 1
        node.n += 1
        node.w += returns[node.player]
        if solved and node.children:
            player = node.children[0].player
            best, all_solved = None, True
            for ch in node.children:
                if ch.outcome is None:
                    all_solved = False
                elif best is None or ch.outcome[player] > best.outcome[player]:
                    best = ch
            if best is not None and (all_solved or
                                     best.outcome[player] == max_utility):
                node.outcome = best.outcome
            else:
                solved = False


def _pruned_visit_counts(root, uct_c, k_forced):
    """KataGo policy-target pruning: remove root-child visits attributable to
    forced playouts / Dirichlet noise, keeping only merit-earned visits (those
    justified by PUCT against the most-visited child). Children left with <=1
    visit are zeroed; the most-visited child keeps everything."""
    kids = root.children
    best = max(kids, key=lambda c: c.n)
    total = sum(c.n for c in kids)
    if total <= 0 or best.n <= 0 or k_forced <= 0:
        return {c.action: c.n for c in kids}
    sqrt_total = math.sqrt(total)
    puct_best = best.w / best.n + uct_c * best.prior * sqrt_total / (1 + best.n)
    out = {}
    for c in kids:
        n = c.n
        if c is best or n == 0:
            out[c.action] = n
            continue
        d = puct_best - c.w / n
        if d <= 0:
            out[c.action] = n
            continue
        n_min    = uct_c * c.prior * sqrt_total / d - 1.0
        n_forced = math.sqrt(k_forced * c.prior * total)
        keep = min(n, max(int(math.ceil(n_min)), n - int(n_forced), 0))
        out[c.action] = 0 if keep <= 1 else keep
    return out


def _solved_adjust_counts(children, legal, counts):
    """See notebook HELPERS cell for the full rationale: MCTS-Solver can prove
    a position solved after exploring just one winning child and correctly
    stop early, leaving raw visit counts almost meaningless. If any legal
    child is a PROVEN WIN for the mover, replace counts with a distribution
    over ONLY the proven-winning actions. Else if some (not all) children are
    PROVEN LOSSES, zero those out. Otherwise return counts unchanged."""
    by_action = {c.action: c for c in children}
    win_mask  = np.zeros(len(legal), dtype=bool)
    loss_mask = np.zeros(len(legal), dtype=bool)
    for i, a in enumerate(legal):
        c = by_action.get(a)
        if c is not None and c.outcome is not None:
            if c.outcome[c.player] == 1.0:
                win_mask[i] = True
            elif c.outcome[c.player] == -1.0:
                loss_mask[i] = True
    if win_mask.any():
        out = np.zeros_like(counts)
        out[win_mask] = 1.0
        return out
    if loss_mask.any() and not loss_mask.all():
        out = counts.copy()
        out[loss_mask] = 0.0
        if out.sum() > 0:
            return out
    return counts


def run_worker(worker_id, req_q, resp_q, pool_resp_q, episode_q, cfg):
    """Pipelined self-play worker. Games are split into two halves that
    leapfrog: while one half's NN request is in flight on the central server,
    the CPU does tree work for the other half. This hides the server round
    trip (queue latency + DirectML dispatch), which otherwise leaves both the
    CPU worker and the GPU idle most of the time."""
    rng = np.random.RandomState(cfg['seed'] + worker_id * 7919)
    random.seed(cfg['seed'] + worker_id * 104729)
    game = pyspiel.load_game('python_boop')
    uct_c = cfg['uct_c']
    k_forced = cfg.get('k_forced', 2.0)
    dirichlet = (cfg['dir_eps'], cfg['dir_alpha'])
    lam = cfg['vt_lambda']

    pool_prob   = cfg.get('pool_prob', 0.0)
    pool_dir    = cfg.get('checkpoint_dir')

    def new_game():
        sims = (cfg['fast_sims'] if random.random() < cfg['fast_prob']
                else cfg['full_sims'])
        slot = {'state': game.new_initial_state(), 'hist': [],
                'move': 0, 'sims': sims, 'root': None, 'pool': None}
        # Opponent diversity: some fraction of games have one side played by a
        # FROZEN historical checkpoint instead of mirror-matching the live net
        # against itself. Only the live side's moves feed the replay buffer
        # (see _resolve_pool_moves) -- this is purely to give the live net a
        # more varied cast of practice opponents and curb self-play strategy
        # cycling, not additional training targets.
        if pool_prob > 0 and pool_dir:
            try:
                labels = [f[6:-3] for f in os.listdir(pool_dir)
                          if f.startswith('bench_') and f.endswith('.pt')]
            except OSError:
                labels = []
            if labels and random.random() < pool_prob:
                slot['pool'] = {'label': random.choice(labels),
                                'side': random.randint(0, 1)}
        return slot

    slots = [new_game() for _ in range(cfg['games_per_worker'])]
    mid = max(1, cfg['games_per_worker'] // 2)
    halves = [list(range(mid)), list(range(mid, cfg['games_per_worker']))]

    def collect(idxs):
        """Gather one leaf wave per game; returns (pending, unique eval list)."""
        pending = []
        for i in idxs:
            s = slots[i]
            pool = s['pool']
            if pool is not None and s['state'].current_player() == pool['side']:
                continue   # next move belongs to the frozen opponent; resolved
                          # by _resolve_pool_moves, not the live-net MCTS tree
            if s['root'] is None:
                s['root'] = _WNode(None, s['state'].current_player(), 1.0)
            root = s['root']
            if root.outcome is not None:
                continue
            wave = 1 if not root.children else min(cfg['wave'],
                                                   s['sims'] - root.n)
            for _ in range(max(wave, 0)):
                if root.outcome is not None:
                    break
                path, st, leaf = _select_leaf(root, s['state'], uct_c, k_forced)
                pending.append((i, path, leaf, st))
        to_eval = {}
        for i, path, leaf, st in pending:
            if not st.is_terminal() and id(leaf) not in to_eval:
                to_eval[id(leaf)] = (i, leaf, st)
        return pending, list(to_eval.values())

    def apply_and_advance(idxs, pending, entries, resp):
        """Expand/backup a completed wave, then finish any completed searches."""
        results = {}
        if entries:
            logits, values = resp
            for (i, leaf, st), lg, val in zip(entries, logits, values):
                is_root = leaf is slots[i]['root']
                _expand(leaf, st.current_player(), st.legal_actions(), lg, rng,
                        dirichlet if is_root else None)
                results[id(leaf)] = (st.current_player(), float(val))
        for i, path, leaf, st in pending:
            if st.is_terminal():
                _backup_terminal(path, st.returns())
            else:
                cur, val = results[id(leaf)]
                _backup_value(path, cur, val)
        for i in idxs:
            s = slots[i]
            root = s['root']
            if root is None or root.n == 0:
                continue
            if root.n < s['sims'] and root.outcome is None:
                continue
            state = s['state']
            player = state.current_player()
            obs = np.asarray(state.observation_tensor(player), dtype=np.float32)
            temperature = 1.0 if s['move'] < cfg['temp_threshold'] else 0.05
            legal = state.legal_actions()
            vis = {c.action: c.n for c in root.children}
            counts = np.array([vis.get(a, 0) for a in legal], dtype=np.float32)
            counts = _solved_adjust_counts(root.children, legal, counts)
            if temperature < 1e-2 or counts.sum() == 0:
                probs = np.zeros(len(legal), dtype=np.float32)
                probs[counts.argmax()] = 1.0
            else:
                log_ct = np.log(counts + 1e-10) / temperature
                log_ct -= log_ct.max()
                ct = np.exp(log_ct)
                probs = ct / ct.sum()
            # Policy TARGET from PRUNED counts (noise-forced visits removed);
            # the move itself is still selected from the raw counts above.
            pvis = _pruned_visit_counts(root, uct_c, k_forced)
            pcounts = np.array([pvis.get(a, 0) for a in legal], dtype=np.float32)
            pcounts = _solved_adjust_counts(root.children, legal, pcounts)
            if pcounts.sum() <= 0:
                pcounts = counts
            visit_dist = (pcounts / pcounts.sum() if pcounts.sum() > 0
                          else np.full(len(legal), 1.0 / len(legal),
                                       dtype=np.float32))
            policy_vec = np.zeros(_NUM_ACTIONS, dtype=np.float32)
            for a, p in zip(legal, visit_dist):
                policy_vec[a] = p
            root_q = root.w / max(root.n, 1)
            s['hist'].append((player, obs, policy_vec, root_q))
            state.apply_action(int(rng.choice(legal, p=probs)))
            s['move'] += 1
            s['root'] = None
            if state.is_terminal():
                returns = state.returns()
                episode_q.put([(o, p, lam * returns[pl] + (1.0 - lam) * q)
                               for pl, o, p, q in s['hist']])
                slots[i] = new_game()

    def _resolve_pool_moves(idxs):
        """For any game in idxs whose CURRENT mover is a designated frozen-
        opponent seat, resolve that single move with ONE direct request to the
        pool net (net_id = its benchmark label) and a greedy (argmax) choice --
        no search tree, since these moves are never used as training targets,
        only to give the live side a more varied opponent to practice
        against. Turns strictly alternate (forced graduation means a player
        is never skipped), so at most one such move is pending per game here.
        """
        for i in idxs:
            s = slots[i]
            pool = s['pool']
            if pool is None:
                continue
            state = s['state']
            if state.current_player() != pool['side']:
                continue
            obs = np.asarray(state.observation_tensor(state.current_player()),
                             dtype=np.float32)
            req_q.put((worker_id, pool['label'], obs[None]))
            logits, _values = pool_resp_q.get()
            legal = state.legal_actions()
            lg = logits[0]
            action = legal[int(np.argmax(lg[legal]))]
            state.apply_action(action)
            s['move'] += 1
            if state.is_terminal():
                returns = state.returns()
                episode_q.put([(o, p, lam * returns[pl] + (1.0 - lam) * q)
                               for pl, o, p, q in s['hist']])
                slots[i] = new_game()

    # Leapfrog loop. Send/receive alternate A,B,A,B so the per-worker FIFO
    # response queue always delivers the response for the half we expect;
    # the `sent` flag keeps alignment when a half had nothing to evaluate.
    inflight = [None, None]          # per half: (pending, entries, sent)
    while True:
        for h in (0, 1):
            if inflight[h] is not None:
                pending, entries, sent = inflight[h]
                resp = resp_q.get() if sent else None
                apply_and_advance(halves[h], pending, entries, resp)
                inflight[h] = None
            _resolve_pool_moves(halves[h])
            pending, entries = collect(halves[h])
            sent = False
            if entries:
                obs = np.stack(
                    [np.asarray(st.observation_tensor(st.current_player()),
                                dtype=np.float32) for _, _, st in entries])
                req_q.put((worker_id, 'live', obs))
                sent = True
            inflight[h] = (pending, entries, sent)
'''

pathlib.Path('boop_mp_worker.py').write_text(_WORKER_SRC, encoding='utf-8')

import multiprocessing as _mp
import threading
import queue as _queue
import time as _time
import os as _os
import sys as _sys


class MPSelfPlayPool:
    """n_workers CPU processes play games; an inference-server thread in this
    process batches ALL their NN requests into single forward passes on
    `device`. Exposes the same .episodes() generator as ParallelSelfPlay.
    `lock` serializes model access between the server thread and training."""

    def __init__(self, network, device, n_workers, cfg, batch_window_s=0.002,
                 checkpoint_dir=None, channels=None, num_blocks=None):
        self.network = network
        self.device = device
        self.checkpoint_dir = checkpoint_dir
        self.channels = channels
        self.num_blocks = num_blocks
        self._pool_nets = {}          # benchmark label -> frozen CPU BoopNet
        self.lock = threading.Lock()
        self._stop = threading.Event()
        ctx = _mp.get_context('spawn')            # Windows-compatible
        self.req_q = ctx.Queue()
        self.episode_q = ctx.Queue(maxsize=64)    # backpressure on workers
        self.resp_qs = [ctx.Queue() for _ in range(n_workers)]
        self.pool_resp_qs = [ctx.Queue() for _ in range(n_workers)]
        self.window = batch_window_s
        # Initialize the autograd engine's per-device state from the MAIN
        # thread before any other thread touches the device: otherwise the
        # first backward races the server thread's device use and trips
        # torch-directml's "device_ready_queues_" INTERNAL ASSERT.
        if str(device) != 'cpu':
            _t = torch.zeros(4, device=device, requires_grad=True)
            (_t * 2.0).sum().backward()
        if _os.getcwd() not in _sys.path:
            _sys.path.insert(0, _os.getcwd())
        import boop_mp_worker
        self.procs = [ctx.Process(target=boop_mp_worker.run_worker,
                                  args=(i, self.req_q, self.resp_qs[i],
                                        self.pool_resp_qs[i],
                                        self.episode_q, cfg), daemon=True)
                      for i in range(n_workers)]
        for p in self.procs:
            p.start()
        self.server = threading.Thread(target=self._serve, daemon=True)
        self.server.start()

    def _get_net(self, net_id):
        """'live' -> the training network on its real device (needs the lock:
        both for DirectML thread-safety and because training mutates its
        weights concurrently). Any other net_id -> a frozen benchmark
        checkpoint, lazily loaded from disk once and cached, always run on
        CPU (no lock needed: never mutated, and CPU torch has no cross-thread
        DirectML hazard). Pool-opponent batches are tiny and infrequent, so
        CPU inference for them costs nothing worth optimizing."""
        if net_id == 'live':
            return self.network, self.device, True
        net = self._pool_nets.get(net_id)
        if net is None:
            path = _os.path.join(self.checkpoint_dir, f'bench_{net_id}.pt')
            net = BoopNet(self.channels, self.num_blocks)
            net.load_state_dict(torch.load(path, map_location='cpu',
                                           weights_only=True))
            net.eval()
            self._pool_nets[net_id] = net
        return net, 'cpu', False

    def _serve(self):
        while not self._stop.is_set():
            try:
                reqs = [self.req_q.get(timeout=0.1)]
            except _queue.Empty:
                continue
            # Short batching window: gather concurrent workers' requests so
            # the GPU sees one big batch instead of several small ones.
            deadline = _time.monotonic() + self.window
            while _time.monotonic() < deadline and len(reqs) < 64:
                try:
                    reqs.append(self.req_q.get_nowait())
                except _queue.Empty:
                    _time.sleep(0.0003)
            # Group by net_id: almost always all 'live', occasionally a request
            # or two tagged with a frozen benchmark label (opponent-diversity
            # self-play games) mixed in.
            groups = {}
            for wid, net_id, obs in reqs:
                groups.setdefault(net_id, []).append((wid, obs))
            for net_id, group in groups.items():
                net, net_device, needs_lock = self._get_net(net_id)
                obs = np.concatenate([o for _, o in group], axis=0)
                xin = _obs_to_9ch(obs)
                # EVERY device op on the LIVE net under the lock (upload,
                # forward, download): DirectML is not safe under concurrent
                # access from two threads. Frozen CPU pool nets need no lock.
                if needs_lock:
                    with self.lock, torch.no_grad():
                        x = torch.from_numpy(xin).to(net_device)
                        logits, values = net(x)
                        lg = logits.cpu().numpy()
                        vv = values.squeeze(-1).cpu().numpy()
                else:
                    with torch.no_grad():
                        x = torch.from_numpy(xin).to(net_device)
                        logits, values = net(x)
                        lg = logits.cpu().numpy()
                        vv = values.squeeze(-1).cpu().numpy()
                off = 0
                target_qs = self.resp_qs if net_id == 'live' else self.pool_resp_qs
                for wid, o in group:
                    n = o.shape[0]
                    target_qs[wid].put((lg[off:off + n], vv[off:off + n]))
                    off += n

    def episodes(self):
        while True:
            yield self.episode_q.get()

    def shutdown(self):
        self._stop.set()
        try:
            self.server.join(timeout=2.0)
        except Exception:
            pass
        for p in self.procs:
            p.terminate()
        for p in self.procs:
            p.join(timeout=2.0)
        for q in [self.req_q, self.episode_q] + self.resp_qs:
            try:
                q.close(); q.cancel_join_thread()
            except Exception:
                pass


In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# Colab GPU estimate: ~1-2 h for 5000 episodes with these settings.
NUM_EPISODES     = 20_000  # run budget only — the LR schedule no longer
                           # depends on it, so extend freely
FULL_MCTS_SIMS   = 1000    # high-quality self-play (25% of games)
FAST_MCTS_SIMS   = 250     # fast self-play (75%) — 2x sims halves the
                           # sampling-noise component of the KL floor
FAST_GAME_PROB   = 0.75    # fraction of episodes using FAST_MCTS_SIMS
# ── Hardware parallelism (AMD GPU via DirectML + 6-core CPU) ─────────────────
USE_WORKERS      = True    # multiprocessing self-play with a central GPU server
SELFPLAY_WORKERS = 4       # CPU game/tree processes (i5-9600K: leave 2 cores for
                           # the main process = GPU server + training)
GAMES_PER_WORKER = 16      # per worker, split into two pipelined halves of 8:
                           # while one half's request is on the GPU, the CPU
                           # does tree work for the other (hides latency)
WORKER_WAVE      = 8       # leaves/game/wave → ~64 obs per request; the server
                           # batches all workers' requests → GPU batches 128-512
N_PARALLEL_GAMES = 8       # (single-process fallback when USE_WORKERS=False)
WAVE_PER_GAME    = 8       # MCTS leaves per game per wave; joint batch ≈ 8×8 = 64
TEMP_THRESHOLD   = 20      # exploratory moves per game — fewer of them
                           # sharpens the game-outcome z in value targets
EVAL_EVERY       = 50      # QUICK eval: live net vs the most recent benchmark
DEEP_EVAL_EVERY  = 500     # DEEP eval: full round-robin tournaments + new benchmark
QUICK_EVAL_GAMES = 20      # games vs the most recent benchmark (quick pulse)
TOURNEY_GAMES_PER_PAIR = 20  # round-robin games per pair (10 each colour)
TOURNEY_FULL_RR  = False   # linear-cost tournaments: new model vs everyone +
                           # REFRESH_PAIRS random old pairs. Full round-robin is
                           # O(pool²) per deep eval — unaffordable as the pool
                           # grows without bound on an indefinite run.
REFRESH_PAIRS    = 4
# Independent evaluation tracks: (name, MCTS simulations per move). Each track
# keeps its OWN Elo table and W/D/L matrix. sims=0 is the raw policy network.
EVAL_TRACKS      = [('policy', 0), ('mcts25', 25), ('mcts50', 50)]
EVAL_TEMP        = 1.0     # (quick-eval pulse only) sampling temperature
EVAL_OPENING_PLIES = 4     # random opening half-moves per eval game (variety, no noise)
START_ELO        = 1000.0  # random and the initial net seed here
K_FACTOR         = 32.0    # Elo update step
BATCH_SIZE       = 512
TRAIN_STEPS_PER_EP = 8     # gradient steps per self-play game
MAX_BUFFER       = 400_000  # richer 200-sim targets are worth a longer window
CHANNELS         = 128
NUM_BLOCKS       = 8
LR_PEAK          = 2e-3
LR_WARMUP_EPS    = 100      # linear warmup episodes
LR_DECAY_EPS     = 12_000   # cosine horizon — decoupled from NUM_EPISODES.
                            # Extending it on a RESUME acts as a warm restart:
                            # the lambda re-evaluates at the resumed step, so
                            # LR pops back up (~1e-3) and decays to the floor
WEIGHT_DECAY     = 1e-4
ENTROPY_BONUS    = 0.01     # policy entropy regularisation coefficient
VALUE_TARGET_LAMBDA = 0.5   # value target = lam*z + (1-lam)*root MCTS value
K_FORCED         = 2.0      # KataGo forced playouts: root children get at
                            # least sqrt(k*P*N) visits during self-play; the
                            # unearned ones are pruned from policy targets
SELFPLAY_POOL_PROB = 0.15   # fraction of self-play games where one random side
                            # is played by a FROZEN benchmark instead of the
                            # live net mirror-matching itself — counters the
                            # non-transitive "strategy cycling" pure self-play
                            # can fall into (only the live side's moves are
                            # ever used as training targets). 0 disables it.
LR_MIN_FACTOR    = 0.10     # cosine decay floor (fraction of LR_PEAK)
CHECKPOINT_DIR   = 'boop_checkpoints_gpu'  # NOT shared with claude/local —
                           # two concurrent runs sharing a dir clobber each other
RESUME_FROM_CHECKPOINT = True  # auto-resume from CHECKPOINT_DIR/latest.pt

# ── Setup ─────────────────────────────────────────────────────────────────────
game         = pyspiel.load_game('python_boop')
base_network = BoopNet(CHANNELS, NUM_BLOCKS).to(DEVICE)   # uncompiled; shares
network      = base_network                               # weights with compiled

# torch.compile() fuses ops and JIT-compiles the network (dynamic=True → one
# graph). Skipped on DirectML (Inductor has no DML backend) and on any setup
# shape-agnostic graph, since MCTS feeds variable batch sizes). It helps on GPU
# but needs a working C++ backend, which native Windows often lacks (Inductor
# raises "Compiler: cl is not found."). torch.compile fails lazily on the FIRST
# forward, so we trigger a dummy forward here and fall back to eager mode if the
# backend can't build. Snapshots deepcopy the uncompiled `base_network` anyway.
# (The batched-leaf MCTS is the main speedup, not compile.)
network = base_network
if _BACKEND != 'directml' and hasattr(torch, 'compile'):
    # Inductor needs a C++ compiler (cl on Windows, g++/gcc/clang elsewhere).
    # Without one the attempt spends ~a minute spamming symbolic-shape warnings
    # before failing anyway — so check first and skip the doomed attempt.
    import shutil, platform, logging
    _cc = (shutil.which('cl') if platform.system() == 'Windows'
           else shutil.which('g++') or shutil.which('gcc') or shutil.which('clang'))
    if _cc is None:
        print('torch.compile: skipped (no C++ compiler for Inductor) — eager mode')
    else:
        # Mute dynamic-shape tracing chatter during compilation.
        logging.getLogger('torch.fx.experimental.symbolic_shapes').setLevel(logging.ERROR)
        try:
            _compiled = torch.compile(base_network, dynamic=True)
            with torch.no_grad():                   # force compilation now
                _compiled(torch.zeros(1, 9, 6, 6, device=DEVICE))
            network = _compiled
            print('torch.compile: enabled')
        except Exception as _e:
            print(f'torch.compile: disabled ({type(_e).__name__}) — running eager mode')

class LerpFreeAdamW(torch.optim.Optimizer):
    """AdamW built from mul_/add_/addcmul_/addcdiv_ only. torch's AdamW calls
    aten::lerp in BOTH its foreach and single-tensor paths, and DirectML lacks
    that op — every optimizer step round-trips each parameter tensor through
    the CPU fallback. Identical math to torch.optim.AdamW (decoupled decay,
    bias correction); verified to track it to float32 precision."""

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps,
                                      weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (b1, b2) = group['lr'], group['betas']
            eps, wd = group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                st = self.state[p]
                if not st:
                    st['step'] = 0
                    st['exp_avg'] = torch.zeros_like(p)
                    st['exp_avg_sq'] = torch.zeros_like(p)
                st['step'] += 1
                t = int(st['step'])
                m, v = st['exp_avg'], st['exp_avg_sq']
                p.mul_(1.0 - lr * wd)                     # decoupled decay
                m.mul_(b1).add_(p.grad, alpha=1.0 - b1)   # lerp-free
                v.mul_(b2).addcmul_(p.grad, p.grad, value=1.0 - b2)
                bc1 = 1.0 - b1 ** t
                bc2 = 1.0 - b2 ** t
                denom = (v.sqrt() / (bc2 ** 0.5)).add_(eps)
                p.addcdiv_(m, denom, value=-(lr / bc1))
        return loss


if _BACKEND == 'directml':
    optimizer = LerpFreeAdamW(network.parameters(),
                              lr=LR_PEAK, weight_decay=WEIGHT_DECAY)
else:
    optimizer = torch.optim.AdamW(network.parameters(),
                                  lr=LR_PEAK, weight_decay=WEIGHT_DECAY)

def _lr_lambda(ep):
    if ep < LR_WARMUP_EPS:
        return ep / max(LR_WARMUP_EPS, 1)
    # Cosine decays over LR_DECAY_EPS (NOT NUM_EPISODES), then holds at the
    # floor — the run can be extended indefinitely without flattening decay.
    frac = min(1.0, (ep - LR_WARMUP_EPS) / max(LR_DECAY_EPS - LR_WARMUP_EPS, 1))
    return max(LR_MIN_FACTOR, 0.5 * (1.0 + np.cos(np.pi * frac)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

# Parallel self-play: N_PARALLEL_GAMES games run concurrently and the MCTS leaf
# waves of ALL of them are evaluated in ONE forward pass (joint batch up to
# N_PARALLEL_GAMES × WAVE_PER_GAME). Weights are read live, so games in flight
# pick up training updates immediately (standard async self-play behaviour).
import os
import threading
try:
    self_play.shutdown()               # re-running this cell: stop old workers
except (NameError, AttributeError):
    pass
if USE_WORKERS:
    _wcfg = dict(seed=int(np.random.randint(1_000_000)), uct_c=1.4,
                 dir_eps=0.25, dir_alpha=0.3, k_forced=K_FORCED,
                 games_per_worker=GAMES_PER_WORKER, wave=WORKER_WAVE,
                 fast_sims=FAST_MCTS_SIMS, full_sims=FULL_MCTS_SIMS,
                 fast_prob=FAST_GAME_PROB, temp_threshold=TEMP_THRESHOLD,
                 vt_lambda=VALUE_TARGET_LAMBDA, pool_prob=SELFPLAY_POOL_PROB,
                 checkpoint_dir=CHECKPOINT_DIR)
    self_play = MPSelfPlayPool(network, DEVICE, SELFPLAY_WORKERS, _wcfg,
                               checkpoint_dir=CHECKPOINT_DIR,
                               channels=CHANNELS, num_blocks=NUM_BLOCKS)
    # Main process only runs the GPU server + training: cap its CPU threads so
    # the self-play workers keep their cores.
    torch.set_num_threads(max(2, (os.cpu_count() or 6) - SELFPLAY_WORKERS))
else:
    self_play = ParallelSelfPlay(game, network, DEVICE,
                                 n_parallel=N_PARALLEL_GAMES,
                                 wave_per_game=WAVE_PER_GAME,
                                 fast_sims=FAST_MCTS_SIMS,
                                 full_sims=FULL_MCTS_SIMS,
                                 fast_prob=FAST_GAME_PROB,
                                 temp_threshold=TEMP_THRESHOLD,
                                 value_target_lambda=VALUE_TARGET_LAMBDA,
                                 pool_prob=SELFPLAY_POOL_PROB,
                                 checkpoint_dir=CHECKPOINT_DIR,
                                 channels=CHANNELS, num_blocks=NUM_BLOCKS)
episode_stream = self_play.episodes()
# Serializes model access between the inference-server thread and training.
MODEL_LOCK = getattr(self_play, 'lock', None) or threading.Lock()

# Small-batch inference (eval games, tournaments) runs on CPU replicas — batch-1
# forwards on DirectML are latency-bound and would dominate eval time.
EVAL_DEVICE = 'cpu'

def _cpu_sd(net):
    return {k: v.detach().cpu() for k, v in net.state_dict().items()}

def _cpu_clone(net):
    m = BoopNet(CHANNELS, NUM_BLOCKS)
    m.load_state_dict(_cpu_sd(net))
    m.eval()
    return m

def _to_cpu_tree(obj):
    if torch.is_tensor(obj):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: _to_cpu_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_cpu_tree(v) for v in obj]
    return obj

replay_buffer = []
# Tournament pool. `random` and the untrained `0` net both start at START_ELO.
_init_snap = _cpu_clone(base_network)
eval_net   = _cpu_clone(base_network)  # refreshed at every eval
benchmarks = [{'label': 'random', 'net': None},
              {'label': '0',      'net': _init_snap}]
# Per-track Elo tables and head-to-head matrices (independent between tracks).
elos = {name: {'random': START_ELO, '0': START_ELO} for name, _ in EVAL_TRACKS}
wdl  = {name: defaultdict(lambda: [0, 0, 0])         for name, _ in EVAL_TRACKS}
hist = {'ep': [], 'p_loss': [], 'v_loss': [], 'kl': [],
        'quick_ep': [], 'q_w': [], 'q_d': [], 'q_l': [],
        'deep_ep': [], 'elo_snap': []}   # elo_snap: list of {track: {label: elo}}

# ── Checkpointing: survive crashes, resume indefinitely ──────────────────────
# Saved at every deep eval: live weights + optimizer/scheduler + every benchmark
# net + elos/wdl/hist. The replay buffer is NOT saved — it refills from fresh
# self-play after a resume. Delete CHECKPOINT_DIR when changing the network
# architecture (old state_dicts won't load into a different net).
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
_LATEST_CKPT = os.path.join(CHECKPOINT_DIR, 'latest.pt')

def _save_benchmark_net(label, net):
    if net is not None:
        torch.save(net.state_dict(),
                   os.path.join(CHECKPOINT_DIR, f'bench_{label}.pt'))

def save_checkpoint(ep):
    # Copy state to CPU first: DirectML (privateuseone) tensors may not pickle.
    with MODEL_LOCK:
        _model_sd = _to_cpu_tree(base_network.state_dict())
        _opt_sd   = _to_cpu_tree(optimizer.state_dict())
    torch.save({'ep': ep,
                'model': _model_sd,
                'optimizer': _opt_sd,
                'scheduler': scheduler.state_dict(),
                'elos': elos,
                'wdl': {name: dict(w) for name, w in wdl.items()},
                'hist': hist,
                'bench_labels': [b['label'] for b in benchmarks]}, _LATEST_CKPT)

start_ep = 1
if RESUME_FROM_CHECKPOINT and os.path.exists(_LATEST_CKPT):
    _ck = torch.load(_LATEST_CKPT, map_location='cpu', weights_only=True)
    base_network.load_state_dict(_ck['model'])
    optimizer.load_state_dict(_ck['optimizer'])
    scheduler.load_state_dict(_ck['scheduler'])
    elos = _ck['elos']
    wdl  = {name: defaultdict(lambda: [0, 0, 0], w)
            for name, w in _ck['wdl'].items()}
    hist = _ck['hist']
    benchmarks = [{'label': 'random', 'net': None}]
    for _lbl in _ck['bench_labels']:
        if _lbl == 'random':
            continue
        _bn = BoopNet(CHANNELS, NUM_BLOCKS)          # benchmarks live on CPU
        _bn.load_state_dict(torch.load(
            os.path.join(CHECKPOINT_DIR, f'bench_{_lbl}.pt'),
            map_location='cpu', weights_only=True))
        _bn.eval()
        benchmarks.append({'label': _lbl, 'net': _bn})
    start_ep = _ck['ep'] + 1
    print(f'Resumed from checkpoint: episode {_ck["ep"]}, pool {len(benchmarks)}')


def _run_all_tracks(focus=None):
    for name, sims in EVAL_TRACKS:
        run_tournament(benchmarks, elos[name], wdl[name], game, EVAL_DEVICE,
                       TOURNEY_GAMES_PER_PAIR, K_FACTOR, EVAL_TEMP, sims,
                       opening_plies=EVAL_OPENING_PLIES,
                       focus_label=focus, refresh_pairs=REFRESH_PAIRS)
    hist['deep_ep'].append(ep_now[0])
    hist['elo_snap'].append({name: dict(elos[name]) for name, _ in EVAL_TRACKS})


def _print_tracks(prefix):
    for name, _ in EVAL_TRACKS:
        ladder = '  '.join(f'{b["label"]}={elos[name][b["label"]]:.0f}'
                           for b in sorted(benchmarks, key=lambda b: -elos[name][b['label']]))
        print(f'{prefix} {name:>7} Elos (strong→weak): {ladder}')


n_params = sum(p.numel() for p in network.parameters() if p.requires_grad)
print(f'Device: {DEVICE}  |  BoopNet params: {n_params:,}')
print(f'{NUM_EPISODES} eps | fast {FAST_MCTS_SIMS} sims ({FAST_GAME_PROB:.0%}) '
      f'/ full {FULL_MCTS_SIMS} sims | {N_PARALLEL_GAMES} games × wave {WAVE_PER_GAME} | 8× aug | {TRAIN_STEPS_PER_EP} steps/ep')
print(f'Quick eval every {EVAL_EVERY} eps (vs latest benchmark); deep tournaments '
      f'every {DEEP_EVAL_EVERY} eps | tracks: {[f"{n}({s})" for n, s in EVAL_TRACKS]}')
print(f'Checkpoints: {CHECKPOINT_DIR} (resume={RESUME_FROM_CHECKPOINT})')
if USE_WORKERS:
    print(f'Self-play: {SELFPLAY_WORKERS} worker processes × {GAMES_PER_WORKER} games '
          f'(wave {WORKER_WAVE}) → GPU inference server on {DEVICE}')
print('-' * 72)

ep_now = [start_ep - 1]
if start_ep == 1:
    # Iteration-0 tournaments: random vs the untrained net, one per track.
    _save_benchmark_net('0', _init_snap)
    _run_all_tracks()
    _print_tracks('Iter     0 |')
    save_checkpoint(0)
print('-' * 72)

for ep in range(start_ep, NUM_EPISODES + 1):
    ep_now[0] = ep

    # 1. Pull the next finished self-play game (all parallel games advance
    #    together; playout-cap randomization is applied per game internally)
    network.eval()
    raw_data = next(episode_stream)

    # 2. Augment each move with all 8 board symmetries (8× free data)
    for obs, pol, val in raw_data:
        replay_buffer.extend(augment_sample(obs, pol, val))
    if len(replay_buffer) > MAX_BUFFER:
        del replay_buffer[:-MAX_BUFFER]

    # 3. Train
    network.train()
    p_losses, v_losses, h_targets = [], [], []
    if len(replay_buffer) >= BATCH_SIZE:
        for _ in range(TRAIN_STEPS_PER_EP):
            batch  = random.sample(replay_buffer, BATCH_SIZE)
            # The ENTIRE step body sits under the lock: every device op —
            # uploads, forward, backward, optimizer, and .item() downloads —
            # must be serialized against the inference-server thread because
            # DirectML is not thread-safe.
            with MODEL_LOCK:
                x_b    = batch_to_tensor([s[0] for s in batch], DEVICE)
                pol_b  = torch.from_numpy(
                             np.stack([s[1] for s in batch]).astype(np.float32)
                         ).to(DEVICE)
                val_b  = torch.from_numpy(
                             np.array([[s[2]] for s in batch], dtype=np.float32)
                         ).to(DEVICE)
                # Target entropy H(pi_target): p_loss - H = KL(target||policy);
                # KL ~ 0 means targets absorbed (raise sims, not params).
                h_targets.append(
                    -(pol_b * torch.log(pol_b + 1e-8)).sum(dim=1).mean().item())

                logits, value = network(x_b)
                log_p  = F.log_softmax(logits, dim=1)

                p_loss = -(pol_b * log_p).sum(dim=1).mean()
                entropy = -(torch.exp(log_p) * log_p).sum(dim=1).mean()
                v_loss = F.mse_loss(value, val_b)
                loss = p_loss - ENTROPY_BONUS * entropy + v_loss

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
                optimizer.step()
                p_losses.append(p_loss.item())
                v_losses.append(v_loss.item())
        scheduler.step()

    if ep % EVAL_EVERY != 0:
        continue

    # 4. Evaluation (CPU replica: batch-1 DirectML inference is latency-bound)
    network.eval()
    with MODEL_LOCK:
        eval_net.load_state_dict(_cpu_sd(base_network))
    mp  = float(np.mean(p_losses)) if p_losses else float('nan')
    mv  = float(np.mean(v_losses)) if v_losses else float('nan')
    mkl = mp - float(np.mean(h_targets)) if h_targets else float('nan')
    hist['ep'].append(ep); hist['p_loss'].append(mp); hist['v_loss'].append(mv)
    hist['kl'].append(mkl)
    lr_now = optimizer.param_groups[0]['lr']

    if ep % DEEP_EVAL_EVERY == 0:
        # DEEP: add current net as a new benchmark, warm-starting each track's Elo
        # from the previous checkpoint, then run one round-robin per track.
        new_label = str(ep)
        for name, _ in EVAL_TRACKS:
            elos[name][new_label] = elos[name][benchmarks[-1]['label']]
        with MODEL_LOCK:
            snap = _cpu_clone(base_network)
        benchmarks.append({'label': new_label, 'net': snap})
        _run_all_tracks(None if TOURNEY_FULL_RR else new_label)
        print(f'Ep {ep:5d} | p={mp:.3f} KL={mkl:.2f} v={mv:.3f} | DEEP tournaments '
              f'(pool {len(benchmarks)}) | lr={lr_now:.2e}')
        _print_tracks('        |')
        _save_benchmark_net(new_label, snap)
        save_checkpoint(ep)
    else:
        # QUICK: policy-only pulse vs the most recent benchmark (no Elo change).
        ref = benchmarks[-1]
        w, d, l = play_match(eval_net, ref['net'], game,
                             QUICK_EVAL_GAMES, EVAL_DEVICE, temp=EVAL_TEMP)
        hist['quick_ep'].append(ep)
        hist['q_w'].append(w); hist['q_d'].append(d); hist['q_l'].append(l)
        print(f'Ep {ep:5d} | p={mp:.3f} KL={mkl:.2f} v={mv:.3f} | vs {ref["label"]:>5} '
              f'W{w:2d}D{d:2d}L{l:2d} (policy Elo={elos["policy"][ref["label"]]:.0f}) '
              f'| lr={lr_now:.2e}')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle('Boop KataGo-Style Training', fontsize=14)

axes[0, 0].plot(hist['ep'], hist['p_loss'], label='CE loss')
axes[0, 0].plot(hist['ep'], hist['kl'], color='tab:green', label='KL(target‖policy)')
axes[0, 0].set_title('Policy: CE and KL — KL→0 means targets are the ceiling')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(hist['ep'], hist['v_loss'], color='tab:orange')
axes[0, 1].set_title('Value Loss (MSE)')
axes[0, 1].set_xlabel('Episode')

# Elo of the model trained-to-episode-N, one line per evaluation track.
# (At each deep eval the newest benchmark's rating under each search budget.)
def _newest_label(ep):
    return '0' if ep == 0 else str(ep)
for name, _ in EVAL_TRACKS:
    xs, ys = [], []
    for ep, snap in zip(hist['deep_ep'], hist['elo_snap']):
        lbl = _newest_label(ep)
        if lbl in snap[name]:
            xs.append(ep); ys.append(snap[name][lbl])
    axes[0, 2].plot(xs, ys, marker='.', label=name)
axes[0, 2].axhline(START_ELO, color='gray', linestyle='--', linewidth=0.8)
axes[0, 2].set_title('Current-model Elo by track')
axes[0, 2].set_xlabel('Episode'); axes[0, 2].legend(fontsize=8)

# Quick-eval progress pulse: policy-only W/D/L vs the most recent benchmark.
axes[1, 0].plot(hist['quick_ep'], hist['q_w'], color='tab:green', marker='.', label='Win')
axes[1, 0].plot(hist['quick_ep'], hist['q_d'], color='gray', linestyle='--', label='Draw')
axes[1, 0].plot(hist['quick_ep'], hist['q_l'], color='tab:red', linestyle=':', label='Loss')
axes[1, 0].set_title(f'Quick eval vs latest benchmark ({QUICK_EVAL_GAMES} games, policy)')
axes[1, 0].set_xlabel('Episode'); axes[1, 0].legend(fontsize=8)

# Final Elo per benchmark, grouped bars by track.
labels = [b['label'] for b in benchmarks]
x = np.arange(len(labels)); width = 0.8 / max(len(EVAL_TRACKS), 1)
for k, (name, _) in enumerate(EVAL_TRACKS):
    vals = [elos[name].get(l, np.nan) for l in labels]
    axes[1, 1].bar(x + (k - (len(EVAL_TRACKS) - 1) / 2) * width, vals, width, label=name)
axes[1, 1].axhline(START_ELO, color='gray', linestyle='--', linewidth=0.8)
axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1, 1].set_title('Final Elo per benchmark'); axes[1, 1].set_ylabel('Elo')
axes[1, 1].legend(fontsize=8)

# Head-to-head win-rate matrix for the strongest track (highest sims).
mtrack = EVAL_TRACKS[-1][0]
W = wdl[mtrack]
n = len(labels)
M = np.full((n, n), np.nan)
for i, ri in enumerate(labels):
    for j, cj in enumerate(labels):
        if (ri, cj) in W:
            w, d, l = W[(ri, cj)]; tot = w + d + l
            if tot: M[i, j] = (w + 0.5 * d) / tot
        elif (cj, ri) in W:
            w, d, l = W[(cj, ri)]; tot = w + d + l
            if tot: M[i, j] = 1.0 - (w + 0.5 * d) / tot
im = axes[1, 2].imshow(M, cmap='RdYlGn', vmin=0, vmax=1)
axes[1, 2].set_xticks(range(n)); axes[1, 2].set_yticks(range(n))
axes[1, 2].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1, 2].set_yticklabels(labels, fontsize=7)
axes[1, 2].set_title(f'Win-rate matrix — {mtrack} (row vs col)')
for i in range(n):
    for j in range(n):
        if not np.isnan(M[i, j]):
            axes[1, 2].text(j, i, f'{M[i, j]:.2f}', ha='center', va='center',
                            fontsize=6, color='black')
fig.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()
